# LeetCode 295 — Find Median from Data Stream bằng Python (Hard)

Notebook này giảng nhập môn một bài **thiết kế class** (design) rất nổi tiếng:

```python
class MedianFinder:

    def __init__(self):


    def addNum(self, num: int) -> None:


    def findMedian(self) -> float:



# Your MedianFinder object will be instantiated and called as such:
# obj = MedianFinder()
# obj.addNum(num)
# param_2 = obj.findMedian()
```

Mục tiêu của bài học:

1. Hiểu **median (trung vị)** thật chắc: số phần tử lẻ thì lấy phần tử giữa, chẵn thì lấy trung bình hai phần tử giữa; tính được bằng tay và bằng code.
2. Hiểu bài **thiết kế class**: object phải **nhớ trạng thái** giữa các lần gọi; đọc được input dạng chuỗi lệnh `["MedianFinder","addNum",...]`, `[[],[1],...]` và tự viết hàm `run_commands` để test giống LeetCode.
3. Đi từ cách người mới tự nghĩ ra (lưu list, mỗi lần hỏi thì sort) → giữ list luôn sắp xếp bằng `bisect.insort` → **hai heap** với `addNum` `O(log n)`, `findMedian` `O(1)`; hiểu rõ mỗi bước cải tiến gỡ được nút thắt nào.
4. Học **heap / `heapq`** từ con số 0: min-heap, nhìn heap dạng cây, vì sao thao tác chỉ tốn `O(log n)`, mẹo **max-heap bằng số âm**.
5. Nắm **2 invariant** (bất biến) của hai heap và giải thích được vì sao thuật toán luôn đúng.
6. Giải cả 2 câu **follow-up**: mảng đếm 101 ô khi mọi số nằm trong `[0, 100]`; thêm phần "ngoài khoảng" khi chỉ 99% số nằm trong `[0, 100]`.
7. Tự kiểm chứng bằng test chéo ngẫu nhiên, tránh các lỗi kinh điển và nhận ra pattern "median / percentile của dòng dữ liệu" ở bài khác.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> **Median** (trung vị) là giá trị nằm ở giữa của một danh sách số nguyên **đã sắp xếp**. Nếu danh sách có số phần tử chẵn thì không có một giá trị chính giữa, khi đó median là **trung bình cộng của hai giá trị ở giữa**.
>
> - Ví dụ, với `arr = [2,3,4]`, median là `3`.
> - Ví dụ, với `arr = [2,3]`, median là `(2 + 3) / 2 = 2.5`.
>
> Hãy cài đặt class `MedianFinder`:
>
> - `MedianFinder()` khởi tạo object `MedianFinder`.
> - `void addNum(int num)` thêm số nguyên `num` từ dòng dữ liệu (data stream) vào cấu trúc dữ liệu.
> - `double findMedian()` trả về median của **tất cả các phần tử đã thêm tính tới lúc này**. Đáp án sai lệch không quá `10^-5` so với đáp án đúng sẽ được chấp nhận.

**Example 1**

```text
Input
["MedianFinder", "addNum", "addNum", "findMedian", "addNum", "findMedian"]
[[], [1], [2], [], [3], []]
Output
[null, null, null, 1.5, null, 2.0]

Explanation
MedianFinder medianFinder = new MedianFinder();
medianFinder.addNum(1);    // arr = [1]
medianFinder.addNum(2);    // arr = [1, 2]
medianFinder.findMedian(); // return 1.5 (i.e., (1 + 2) / 2)
medianFinder.addNum(3);    // arr = [1, 2, 3]
medianFinder.findMedian(); // return 2.0
```

**Constraints**

- `-10^5 <= num <= 10^5`
- Luôn có ít nhất một phần tử trong cấu trúc dữ liệu trước khi gọi `findMedian`.
- Có tối đa `5 * 10^4` lời gọi `addNum` và `findMedian` (tính chung cả hai).

**Follow up**

- Nếu **mọi** số trong stream đều nằm trong khoảng `[0, 100]`, bạn sẽ tối ưu lời giải thế nào?
- Nếu **99%** số trong stream nằm trong khoảng `[0, 100]`, bạn sẽ tối ưu lời giải thế nào?

### Ký hiệu dùng suốt notebook

- `n` = số phần tử đã được `addNum` tính tới thời điểm đang xét.
- `q` = tổng số lời gọi `addNum` + `findMedian` (theo đề `q <= 5·10^4`).

### 1.1 Đọc input / output của bài "thiết kế class"

Đây **không** phải dạng "cho mảng, trả kết quả". LeetCode mô tả một **kịch bản gọi hàm** bằng 2 list song song: list thứ nhất là **tên lệnh**, list thứ hai là **tham số** của lệnh tương ứng.

| Vị trí | Tên lệnh | Tham số | Nghĩa là | Output |
|---:|---|---|---|---|
| 0 | `"MedianFinder"` | `[]` | `obj = MedianFinder()` | `null` |
| 1 | `"addNum"` | `[1]` | `obj.addNum(1)` | `null` (hàm `void` không trả gì) |
| 2 | `"addNum"` | `[2]` | `obj.addNum(2)` | `null` |
| 3 | `"findMedian"` | `[]` | `obj.findMedian()` khi đã có `[1, 2]` | `1.5` |
| 4 | `"addNum"` | `[3]` | `obj.addNum(3)` | `null` |
| 5 | `"findMedian"` | `[]` | `obj.findMedian()` khi đã có `[1, 2, 3]` | `2.0` |

- `null` là cách Java / JSON viết "không có giá trị"; trong Python tương ứng là `None`.
- Hai lời gọi `findMedian` hỏi về **hai thời điểm khác nhau**: lần đầu dữ liệu chỉ có `[1, 2]`, lần sau có `[1, 2, 3]`. Vì vậy object phải **nhớ** mọi số đã thêm.

### Vì sao bài này là Hard?

Nếu chỉ hỏi median **một lần** ở cuối thì rất dễ: sắp xếp rồi lấy phần tử giữa. Cái khó là dữ liệu **chảy vào từng số một** (data stream — dòng dữ liệu), còn `findMedian` có thể bị hỏi **bất cứ lúc nào**, tổng cộng tới `5·10^4` lời gọi. Ta phải thiết kế cấu trúc dữ liệu sao cho **cả hai** thao tác đều nhanh:

- `addNum` nhanh → đừng làm việc nặng mỗi lần thêm.
- `findMedian` nhanh → đừng làm việc nặng mỗi lần hỏi.

Hai mong muốn này "kéo co" với nhau. Notebook sẽ đi từng bước để tìm ra điểm cân bằng đẹp: `addNum` tốn `O(log n)`, `findMedian` tốn `O(1)`.

## 2. Nền tảng

### 2.1 Median là gì? Tính thế nào khi số phần tử lẻ / chẵn?

Quy trình luôn gồm 2 bước: **(1) sắp xếp tăng dần, (2) lấy phần tử ở giữa.**

- Lẻ: `[5, 1, 3]` → sắp xếp `[1, 3, 5]` → phần tử giữa là `3` → median = `3`.
- Chẵn: `[8, 1, 5, 3]` → sắp xếp `[1, 3, 5, 8]` → hai phần tử giữa là `3` và `5` → median = `(3 + 5) / 2 = 4.0`.

Trong Python, chỉ số (index) bắt đầu từ `0`. Với list `a` đã sắp xếp, dài `n`:

| `n` | Lẻ hay chẵn? | Chỉ số ở giữa | Median |
|---:|---|---|---|
| `5` | lẻ (`5 % 2 == 1`) | `n // 2 = 2` | `a[2]` |
| `4` | chẵn (`4 % 2 == 0`) | `n // 2 - 1 = 1` và `n // 2 = 2` | `(a[1] + a[2]) / 2` |

Nhắc lại 2 phép toán:

- `n // 2` là **chia lấy phần nguyên**: `5 // 2 = 2`, `4 // 2 = 2`.
- `n % 2` là **lấy phần dư**: `5 % 2 = 1` (lẻ), `4 % 2 = 0` (chẵn).

Vì sao với `n` lẻ thì phần tử giữa nằm ở chỉ số `n // 2`? Vì trước nó có đúng `n // 2` phần tử và sau nó cũng có `n // 2` phần tử. Ví dụ `n = 5`: trước chỉ số `2` có 2 phần tử (chỉ số `0`, `1`), sau nó có 2 phần tử (chỉ số `3`, `4`).

In [1]:
def median_of_sorted(a):
    # a là list ĐÃ sắp xếp tăng dần và không rỗng
    n = len(a)
    mid = n // 2
    if n % 2 == 1:                      # n lẻ: có đúng 1 phần tử giữa
        return float(a[mid])
    return (a[mid - 1] + a[mid]) / 2    # n chẵn: trung bình 2 phần tử giữa


for arr in [[2, 3, 4], [2, 3], [1, 3, 5, 8, 15], [1, 3, 5, 8], [7], [-5, -1]]:
    n = len(arr)
    kind = "lẻ" if n % 2 == 1 else "chẵn"
    print(f"{str(arr):<18} n = {n} ({kind:<4}) -> median = {median_of_sorted(arr)}")

[2, 3, 4]          n = 3 (lẻ  ) -> median = 3.0
[2, 3]             n = 2 (chẵn) -> median = 2.5
[1, 3, 5, 8, 15]   n = 5 (lẻ  ) -> median = 5.0
[1, 3, 5, 8]       n = 4 (chẵn) -> median = 4.0
[7]                n = 1 (lẻ  ) -> median = 7.0
[-5, -1]           n = 2 (chẵn) -> median = -3.0


### 2.2 Bốn chi tiết nhỏ hay làm sai

1. **Phải sắp xếp trước.** Với `[5, 1, 3]`, nếu lấy ngay phần tử ở giữa sẽ ra `1` — sai; đúng phải là `3`.
2. **`/` khác `//`.** Phép `/` luôn cho số thực (float): `(1 + 2) / 2 = 1.5`, `4 / 2 = 2.0`. Phép `//` chia rồi **làm tròn xuống** (floor): `(1 + 2) // 2 = 1` → mất phần `.5`. Với số âm càng dễ bẫy: `(-1 + -2) // 2 = -2` (làm tròn xuống về phía âm vô cùng), trong khi median đúng là `-1.5`.
3. **Kiểu trả về là `float`.** LeetCode in `2.0`. Trong Python, `float(3)` cho `3.0`. LeetCode chấp nhận sai số `10^-5` nên trả `2` hay `2.0` đều qua, nhưng ta luôn trả `float` cho khớp chữ ký `-> float` — đó là thói quen tốt.
4. **Median khác trung bình cộng (mean).** Với `[1, 2, 100]`: mean ≈ `34.33` còn median = `2`. Median "không bị kéo" bởi một giá trị cực lớn, vì vậy nó được dùng nhiều trong thống kê (thu nhập, độ trễ mạng, thời gian phản hồi...).

In [2]:
data = [5, 1, 3]
print("Lấy giữa khi CHƯA sort:", data[len(data) // 2])           # 1  -> sai
print("Lấy giữa sau khi sort  :", sorted(data)[len(data) // 2])   # 3  -> đúng

print((1 + 2) / 2, (1 + 2) // 2)        # 1.5 1
print((-1 + -2) / 2, (-1 + -2) // 2)    # -1.5 -2   <- // làm tròn XUỐNG
print(4 / 2, float(3))                  # 2.0 3.0

nums = [1, 2, 100]
print("mean =", round(sum(nums) / len(nums), 2), "| median =", median_of_sorted(sorted(nums)))

Lấy giữa khi CHƯA sort: 1
Lấy giữa sau khi sort  : 3
1.5 1
-1.5 -2
2.0 3.0
mean = 34.33 | median = 2.0


### 2.3 Class và trạng thái: object "nhớ" dữ liệu giữa các lần gọi

Một **class** là "bản thiết kế"; mỗi lần gọi `MedianFinder()` ta tạo ra một **object** (đối tượng) mới theo bản thiết kế đó.

- `__init__(self)` chạy **đúng 1 lần** khi tạo object. Đây là chỗ tạo "ngăn chứa" dữ liệu.
- `self` là chính object đang được dùng. Viết `self.count = 0` nghĩa là "object này có một ngăn tên `count`, đang chứa 0".
- Các method khác (`addNum`, `findMedian`) đọc / sửa các ngăn `self.xxx`. Vì ngăn nằm trên object nên giá trị **còn nguyên** giữa các lần gọi. Biến cục bộ (không có `self.`) thì biến mất khi method chạy xong.

Ví dụ mini trước khi vào bài: class `RunningMean` tính **trung bình cộng** của một dòng dữ liệu. Nó chỉ cần nhớ **2 con số**: tổng (`total`) và số lượng (`count`).

Hãy để ý một điều quan trọng: với **trung bình cộng**, 2 con số là đủ. Với **median** thì không: thêm một số mới có thể làm median "nhảy" sang một phần tử khác hẳn, nên ta phải biết **thứ tự** của dữ liệu. Đó là lý do bài median khó hơn nhiều.

In [3]:
class RunningMean:
    def __init__(self):          # chạy 1 lần khi tạo object
        self.count = 0           # ngăn "count" của riêng object này
        self.total = 0

    def add(self, x):
        self.count += 1
        self.total += x

    def mean(self):
        return self.total / self.count


a = RunningMean()
a.add(4)
a.add(6)
print("a.mean() =", a.mean())      # (4 + 6) / 2 = 5.0
a.add(11)
print("a.mean() =", a.mean())      # (4 + 6 + 11) / 3 = 7.0

b = RunningMean()                  # object MỚI -> trạng thái mới, không dính dữ liệu của a
b.add(100)
print("b.mean() =", b.mean(), "| a vẫn là", a.mean())

a.mean() = 5.0
a.mean() = 7.0
b.mean() = 100.0 | a vẫn là 7.0


### 2.4 Tự viết hàm chạy chuỗi lệnh kiểu LeetCode

Để test giống hệt LeetCode, ta viết hàm `run_commands(cls, ops, args)`:

1. Tạo object: `obj = cls(*args[0])`.
2. Với mỗi cặp (`op`, `a`) còn lại: lấy method có tên `op` bằng `getattr(obj, op)` rồi gọi nó với tham số `*a`.
3. Gom kết quả từng lời gọi vào list `outputs` (lời gọi khởi tạo cho `None`, tương ứng `null`).

Hai công cụ Python mới:

- `getattr(obj, "add")` lấy method (hoặc thuộc tính) của object theo **tên dạng chuỗi**. `getattr(rm, "add")(10)` giống hệt `rm.add(10)`. Ta cần nó vì `ops` chứa tên lệnh ở dạng chuỗi như `"addNum"`, `"findMedian"`.
- `*a` khi gọi hàm: "rải" (unpack) list thành các đối số riêng lẻ. `f(*[1])` giống `f(1)`, còn `f(*[])` giống `f()`. Nhờ vậy cùng một dòng code gọi được cả `addNum(1)` (1 tham số) lẫn `findMedian()` (0 tham số).

Thêm 2 hàm phụ cho tiện:

- `show(outputs)` in kết quả giống LeetCode (`None` → `null`).
- `build_commands(tokens)` giúp viết test gọn: số nguyên → `addNum(số)`, chuỗi `"?"` → `findMedian()`. Ví dụ `[1, 2, "?", 3, "?"]` chính là Example 1.

Lưu ý: `ops[0]` chỉ báo "tạo object". Ta truyền **class thật** vào tham số `cls`, nhờ vậy chạy được nhiều cách cài đặt khác nhau (`MedianFinderSortEachTime`, `MedianFinderInsort`, ...) trên **cùng một** chuỗi lệnh.

In [4]:
rm = RunningMean()
getattr(rm, "add")(10)             # giống rm.add(10)
getattr(rm, "add")(*[20])          # *[20] rải list thành 1 đối số: rm.add(20)
print(getattr(rm, "mean")(*[]))    # *[] rải ra 0 đối số: rm.mean() -> 15.0


def run_commands(cls, ops, args):
    # Chạy chuỗi lệnh kiểu LeetCode: ops[0] là lệnh tạo object, args[0] là tham số của __init__
    obj = cls(*args[0])
    outputs = [None]                        # lời gọi khởi tạo -> null
    for op, a in zip(ops[1:], args[1:]):
        method = getattr(obj, op)           # lấy method theo tên, ví dụ "addNum"
        outputs.append(method(*a))          # gọi với tham số rải từ list a
    return outputs


def show(outputs):
    # In giống LeetCode: None -> null
    return "[" + ", ".join("null" if x is None else str(x) for x in outputs) + "]"


def build_commands(tokens, class_name="MedianFinder"):
    # Số nguyên -> addNum(số); "?" -> findMedian()
    ops, args = [class_name], [[]]
    for t in tokens:
        if t == "?":
            ops.append("findMedian")
            args.append([])
        else:
            ops.append("addNum")
            args.append([t])
    return ops, args


ops = ["RunningMean", "add", "add", "mean", "add", "mean"]
args = [[], [4], [6], [], [11], []]
print(show(run_commands(RunningMean, ops, args)))      # [null, null, null, 5.0, null, 7.0]

ops1, args1 = build_commands([1, 2, "?", 3, "?"])        # Example 1 của đề
print(ops1)
print(args1)
assert ops1 == ["MedianFinder", "addNum", "addNum", "findMedian", "addNum", "findMedian"]
assert args1 == [[], [1], [2], [], [3], []]

15.0
[null, null, null, 5.0, null, 7.0]
['MedianFinder', 'addNum', 'addNum', 'findMedian', 'addNum', 'findMedian']
[[], [1], [2], [], [3], []]


## 3. Cách 1 — Lưu hết vào list, mỗi lần hỏi thì sort

Người mới thường nghĩ ngay: *"Cứ lưu mọi số lại. Khi nào bị hỏi median thì sắp xếp rồi lấy phần tử giữa."* Đây đúng là định nghĩa median, nên chắc chắn đúng.

- `__init__`: tạo `self.nums = []`.
- `addNum(num)`: `self.nums.append(num)` — thêm vào cuối list, tốn `O(1)`.
- `findMedian()`: `sorted(self.nums)` tạo bản sao đã sắp xếp, rồi gọi `median_of_sorted`.

Ta chạy Example 1 và một chuỗi "khó" hơn: thêm lần lượt `5, 15, 1, 3, 8`, hỏi median sau **mỗi** lần thêm. Chuỗi này sẽ đi theo ta suốt notebook.

Kết quả mong đợi của chuỗi `5, 15, 1, 3, 8` (tính tay):

| Sau khi thêm | Dữ liệu đã sắp xếp | Median |
|---|---|---:|
| `5` | `[5]` | `5.0` |
| `15` | `[5, 15]` | `10.0` |
| `1` | `[1, 5, 15]` | `5.0` |
| `3` | `[1, 3, 5, 15]` | `4.0` |
| `8` | `[1, 3, 5, 8, 15]` | `5.0` |

In [5]:
class MedianFinderSortEachTime:
    def __init__(self):
        self.nums = []                                 # lưu mọi số theo thứ tự đến

    def addNum(self, num: int) -> None:
        self.nums.append(num)                          # O(1)

    def findMedian(self) -> float:
        return median_of_sorted(sorted(self.nums))     # sort lại TỪ ĐẦU mỗi lần hỏi


print("Example 1 :", show(run_commands(MedianFinderSortEachTime, ops1, args1)))

ops2, args2 = build_commands([5, "?", 15, "?", 1, "?", 3, "?", 8, "?"])
print("5,15,1,3,8:", show(run_commands(MedianFinderSortEachTime, ops2, args2)))

Example 1 : [null, null, null, 1.5, null, 2.0]
5,15,1,3,8: [null, null, 5.0, null, 10.0, null, 5.0, null, 4.0, null, 5.0]


### Độ phức tạp cách 1 và chỗ vướng

- `addNum`: `O(1)`.
- `findMedian`: sắp xếp `n` số tốn `O(n log n)`.
- Bộ nhớ: `O(n)`.

Trường hợp xấu: `5·10^4` lời gọi, `addNum` và `findMedian` xen kẽ nhau → khoảng `25 000` lần hỏi. Lần hỏi thứ `i` phải sắp xếp `i` số, nên tổng số phần tử bị đem đi sắp xếp là `1 + 2 + ... + 25000 = 25000·25001/2 ≈ 3.1·10^8`; nhân thêm hệ số `log n` của việc sắp xếp, ta ra cỡ **vài tỷ** phép so sánh → gần như chắc chắn **TLE** (Time Limit Exceeded — quá thời gian cho phép). Mục 13 sẽ in bảng ước lượng cụ thể.

**Nút thắt:** mỗi lần hỏi ta **sắp xếp lại từ đầu**, trong khi giữa hai lần hỏi dữ liệu chỉ thêm đúng 1 số. Toàn bộ công sức "đã sắp xếp" của lần trước bị vứt đi.

(Chi tiết thêm: nếu sort tại chỗ bằng `self.nums.sort()`, list sẽ "gần như đã sắp xếp" ở lần sau, và thuật toán Timsort của Python tận dụng được điều này nên chạy nhanh hơn, cỡ `O(n)` mỗi lần. Nhưng như vậy vẫn là ít nhất `O(n)` cho **mỗi** lần hỏi, tổng vẫn `O(q^2)`.)

**Ý tưởng cải tiến:** giữ list **luôn luôn sắp xếp** ngay từ lúc thêm: chèn số mới vào **đúng chỗ** của nó. Khi đó `findMedian` chỉ cần đọc phần tử giữa. Muốn tìm "đúng chỗ" thật nhanh trong list đã sắp xếp → dùng **binary search** (tìm kiếm nhị phân).

## 4. Nền tảng: Binary search và module `bisect`

### 4.1 Binary search tìm vị trí chèn

Bài toán nhỏ: cho list **đã sắp xếp** `a = [1, 3, 5, 8, 15, 20, 27]` và số `x = 9`. Chèn `x` vào vị trí nào để list vẫn sắp xếp?

Cách dò từng phần tử từ trái sang tốn `O(n)`. **Binary search** làm khôn hơn: nhìn phần tử **ở giữa** đoạn đang xét rồi bỏ đi một nửa.

- Đoạn đang xét là `a[lo:hi]`, ban đầu `lo = 0`, `hi = 7`.
- `mid = (lo + hi) // 2 = 3`, `a[3] = 8 < 9` → vị trí chèn phải nằm **bên phải** `mid` → `lo = 4`.
- `mid = (4 + 7) // 2 = 5`, `a[5] = 20 >= 9` → vị trí chèn nằm ở `mid` hoặc **bên trái** → `hi = 5`.
- `mid = (4 + 5) // 2 = 4`, `a[4] = 15 >= 9` → `hi = 4`.
- `lo == hi == 4` → dừng: chèn `9` vào vị trí `4` (ngay trước `15`).

Chỉ 3 lần so sánh cho 7 phần tử. Mỗi bước đoạn tìm kiếm còn một nửa, nên với `50 000` phần tử chỉ cần khoảng `16` lần so sánh: đó là `O(log n)`.

### 4.2 Module `bisect` của thư viện chuẩn

Python có sẵn module `bisect` làm đúng việc trên:

| Hàm | Trả về / làm gì |
|---|---|
| `bisect.bisect_left(a, x)` | vị trí chèn `x` **bên trái** các số bằng `x` |
| `bisect.bisect_right(a, x)` | vị trí chèn `x` **bên phải** các số bằng `x` |
| `bisect.insort(a, x)` | tìm vị trí bằng `bisect_right` rồi **chèn luôn** `x` vào `a` |

Cell dưới tự cài `my_bisect_left` có in từng bước (để thấy binary search chạy), rồi so với module `bisect`.

In [6]:
import bisect


def my_bisect_left(a, x, verbose=False):
    lo, hi = 0, len(a)                      # tìm trong đoạn a[lo:hi]
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            if verbose:
                print(f"  lo={lo}, hi={hi}, mid={mid}: a[mid]={a[mid]} < {x}  -> bỏ nửa trái, lo = {mid + 1}")
            lo = mid + 1
        else:
            if verbose:
                print(f"  lo={lo}, hi={hi}, mid={mid}: a[mid]={a[mid]} >= {x} -> bỏ nửa phải, hi = {mid}")
            hi = mid
    return lo


a = [1, 3, 5, 8, 15, 20, 27]
print("Tìm vị trí chèn 9 vào", a)
pos = my_bisect_left(a, 9, verbose=True)
print("-> vị trí", pos, "| bisect_left của Python:", bisect.bisect_left(a, 9))

b = [1, 3, 5, 5, 8]
print("\nb =", b)
print("bisect_left(b, 5)  =", bisect.bisect_left(b, 5))     # 2: trước hai số 5
print("bisect_right(b, 5) =", bisect.bisect_right(b, 5))    # 4: sau hai số 5
print("bisect_left(b, 0), bisect_left(b, 9) =", bisect.bisect_left(b, 0), bisect.bisect_left(b, 9))
bisect.insort(b, 4)
print("sau insort(b, 4):", b)

Tìm vị trí chèn 9 vào [1, 3, 5, 8, 15, 20, 27]
  lo=0, hi=7, mid=3: a[mid]=8 < 9  -> bỏ nửa trái, lo = 4
  lo=4, hi=7, mid=5: a[mid]=20 >= 9 -> bỏ nửa phải, hi = 5
  lo=4, hi=5, mid=4: a[mid]=15 >= 9 -> bỏ nửa phải, hi = 4
-> vị trí 4 | bisect_left của Python: 4

b = [1, 3, 5, 5, 8]
bisect_left(b, 5)  = 2
bisect_right(b, 5) = 4
bisect_left(b, 0), bisect_left(b, 9) = 0 5
sau insort(b, 4): [1, 3, 4, 5, 5, 8]


### 4.3 Tìm vị trí thì nhanh, nhưng **chèn** vào list tốn bao nhiêu?

Python list lưu các phần tử **liền nhau** trong bộ nhớ (ô 0, ô 1, ô 2, ...). Muốn chèn `3` vào vị trí `1` của `[1, 5, 8, 15]`, mọi phần tử từ vị trí `1` trở đi phải **dời sang phải 1 ô** để lấy chỗ trống:

```text
trước :  [1, 5, 8, 15]
dời   :  [1, _, 5, 8, 15]    <- 15, 8, 5 lần lượt dời sang phải (3 lần dời)
đặt 3 :  [1, 3, 5, 8, 15]
```

Vậy `bisect.insort` gồm 2 phần:

- tìm vị trí bằng binary search: `O(log n)`;
- dời các phần tử phía sau: `O(n)` (xấu nhất khi chèn vào **đầu** list: dời cả `n` phần tử).

Tổng cộng: **`O(n)`** mỗi lần chèn. Cell dưới đếm số lần dời cho vài trường hợp.

In [7]:
def insert_and_count_shifts(arr, x):
    pos = bisect.bisect_right(arr, x)       # vị trí chèn (giống insort)
    shifts = len(arr) - pos                 # số phần tử phải dời sang phải 1 ô
    arr.insert(pos, x)
    return pos, shifts


a = [1, 5, 8, 15]
print("chèn 3   -> (vị trí, số lần dời) =", insert_and_count_shifts(a, 3), a)
print("chèn 100 -> (vị trí, số lần dời) =", insert_and_count_shifts(a, 100), a)   # chèn cuối: dời 0
print("chèn -7  -> (vị trí, số lần dời) =", insert_and_count_shifts(a, -7), a)    # chèn đầu: dời tất cả

big = list(range(1, 50001))
print("chèn 0 vào list 50000 số -> (vị trí, số lần dời) =", insert_and_count_shifts(big, 0))

chèn 3   -> (vị trí, số lần dời) = (1, 3) [1, 3, 5, 8, 15]
chèn 100 -> (vị trí, số lần dời) = (5, 0) [1, 3, 5, 8, 15, 100]
chèn -7  -> (vị trí, số lần dời) = (0, 6) [-7, 1, 3, 5, 8, 15, 100]
chèn 0 vào list 50000 số -> (vị trí, số lần dời) = (0, 50000)


## 5. Cách 2 — Giữ list luôn sắp xếp bằng `bisect.insort`

Cải tiến từ cách 1: thay vì "thêm bừa vào cuối, hỏi thì sort", ta **chèn đúng chỗ ngay khi thêm**.

- `__init__`: `self.data = []` — list này **luôn** được giữ sắp xếp tăng dần (đây là invariant của cách 2).
- `addNum(num)`: `bisect.insort(self.data, num)`.
- `findMedian()`: list đã sắp xếp sẵn → gọi thẳng `median_of_sorted(self.data)`, chỉ đọc 1–2 phần tử: `O(1)`.

Hàm `trace_insort` in ra: vị trí chèn, số phần tử phải dời, list sau khi chèn và median ngay sau đó.

In [8]:
class MedianFinderInsort:
    def __init__(self):
        self.data = []                          # LUÔN sắp xếp tăng dần

    def addNum(self, num: int) -> None:
        bisect.insort(self.data, num)           # tìm vị trí O(log n) + dời O(n)

    def findMedian(self) -> float:
        return median_of_sorted(self.data)      # không cần sort lại: O(1)


print("Example 1:", show(run_commands(MedianFinderInsort, ops1, args1)))


def trace_insort(stream):
    data = []
    for num in stream:
        pos = bisect.bisect_right(data, num)
        shifts = len(data) - pos
        bisect.insort(data, num)
        print(f"thêm {num:>2}: vị trí {pos}, dời {shifts} phần tử -> {str(data):<18} median = {median_of_sorted(data)}")


trace_insort([5, 15, 1, 3, 8])

Example 1: [null, null, null, 1.5, null, 2.0]
thêm  5: vị trí 0, dời 0 phần tử -> [5]                median = 5.0
thêm 15: vị trí 1, dời 0 phần tử -> [5, 15]            median = 10.0
thêm  1: vị trí 0, dời 2 phần tử -> [1, 5, 15]         median = 5.0
thêm  3: vị trí 1, dời 2 phần tử -> [1, 3, 5, 15]      median = 4.0
thêm  8: vị trí 3, dời 1 phần tử -> [1, 3, 5, 8, 15]   median = 5.0


### Độ phức tạp cách 2 và chỗ vướng

- `addNum`: `O(log n)` để tìm + `O(n)` để dời → **`O(n)`**.
- `findMedian`: **`O(1)`**.
- Bộ nhớ: `O(n)`.

So với cách 1, ta đã chuyển việc nặng từ lúc **hỏi** sang lúc **thêm**, và việc nặng nhẹ đi nhiều (dời tối đa `n` phần tử thay vì sắp xếp `n log n`). Nhưng trường hợp xấu vẫn là `O(q^2)` tổng cộng — cỡ `10^8` lần dời cho `5·10^4` lời gọi.

Trên thực tế cách này **thường vẫn chạy kịp** trên LeetCode, vì việc dời phần tử được Python làm bằng C (chép cả khối bộ nhớ một lần) nên rất nhanh. Tuy vậy, về Big-O nó vẫn là `O(n)` mỗi lần thêm, và người phỏng vấn chắc chắn sẽ hỏi tiếp: *"Làm tốt hơn được không?"*

### Câu hỏi then chốt: ta có thật sự cần **cả list** được sắp xếp?

Nhìn lại `median_of_sorted`: nó chỉ đọc `a[mid - 1]` và `a[mid]` — **1 hoặc 2 phần tử ở giữa**. Ta đang trả giá `O(n)` để giữ đúng thứ tự của **mọi** phần tử, trong khi thứ thật sự cần chỉ là:

- **số lớn nhất của nửa nhỏ**, và
- **số nhỏ nhất của nửa lớn**.

Có một cấu trúc dữ liệu chuyên trả lời câu hỏi "ai lớn nhất / nhỏ nhất?" trong `O(1)` và cho thêm phần tử trong `O(log n)` mà **không** cần giữ thứ tự toàn bộ: đó là **heap**. Ta học heap ngay sau đây.

## 6. Nền tảng: Heap và module `heapq`

### 6.1 Heap là gì?

Hình dung hàng chờ ưu tiên ở phòng cấp cứu: người **nặng nhất** luôn được gọi trước; người mới đến được xếp vào đúng mức ưu tiên, nhưng không ai phải xếp **cả hàng** theo thứ tự hoàn hảo. Heap là cấu trúc dữ liệu làm đúng việc đó.

**Min-heap** là một cây nhị phân (mỗi node có tối đa 2 con) thỏa **tính chất heap**: giá trị ở **cha ≤ giá trị ở mỗi con**. Hệ quả: phần tử **nhỏ nhất luôn nằm ở gốc**.

Python không tạo cây bằng object; nó lưu heap trong **một list thường** theo quy ước vị trí:

- gốc ở `h[0]`;
- hai con của `h[i]` là `h[2*i + 1]` và `h[2*i + 2]`; cha của `h[i]` là `h[(i - 1) // 2]`.

Module `heapq` (thư viện chuẩn) cung cấp các thao tác:

| Lệnh | Làm gì | Time (heap có `m` phần tử) |
|---|---|---|
| `heapq.heappush(h, x)` | thêm `x` vào heap | `O(log m)` |
| `heapq.heappop(h)` | lấy ra **và xóa** phần tử nhỏ nhất | `O(log m)` |
| `h[0]` | **xem** phần tử nhỏ nhất, không xóa ("peek") | `O(1)` |
| `heapq.heapify(h)` | biến một list bất kỳ thành heap tại chỗ | `O(m)` |
| `len(h)` | số phần tử | `O(1)` |

Lưu ý quan trọng: heap **không phải** list đã sắp xếp. Nó chỉ cam kết `h[0]` là nhỏ nhất. Chính vì "cam kết ít hơn" nên nó rẻ hơn việc giữ cả list sắp xếp.

In [9]:
import heapq

h = []
for x in [5, 1, 8, 3, 2]:
    heapq.heappush(h, x)
    print(f"push {x}: list bên trong = {str(h):<16} -> nhỏ nhất h[0] = {h[0]}")

print("\nPop lần lượt:", [heapq.heappop(h) for _ in range(5)], "| heap còn:", h)

arr = [9, 4, 7, 1, 6]
heapq.heapify(arr)
print("heapify [9, 4, 7, 1, 6] ->", arr, "| nhỏ nhất =", arr[0])

push 5: list bên trong = [5]              -> nhỏ nhất h[0] = 5
push 1: list bên trong = [1, 5]           -> nhỏ nhất h[0] = 1
push 8: list bên trong = [1, 5, 8]        -> nhỏ nhất h[0] = 1
push 3: list bên trong = [1, 3, 8, 5]     -> nhỏ nhất h[0] = 1
push 2: list bên trong = [1, 2, 8, 5, 3]  -> nhỏ nhất h[0] = 1

Pop lần lượt: [1, 2, 3, 5, 8] | heap còn: []
heapify [9, 4, 7, 1, 6] -> [1, 4, 7, 9, 6] | nhỏ nhất = 1


Quan sát:

- Sau 5 lần push, list bên trong là `[1, 2, 8, 5, 3]` — **không** sắp xếp, nhưng `h[0] = 1` luôn là nhỏ nhất.
- Pop lần lượt ra `[1, 2, 3, 5, 8]` — tăng dần. Có thể nghĩ heap là một "máy rút số nhỏ nhất".

### 6.2 Nhìn heap dạng cây

List `[1, 2, 8, 5, 3]` thật ra là cây sau (đọc theo từng tầng, từ trái sang phải):

```text
            1            <- tầng 0: h[0]
         /     \
        2       8        <- tầng 1: h[1], h[2]
       / \
      5   3              <- tầng 2: h[3], h[4]  (con của h[1] = 2)
```

Kiểm tra tính chất heap: `1 ≤ 2`, `1 ≤ 8`, `2 ≤ 5`, `2 ≤ 3` — đúng hết. Còn `8` lớn hơn `5` và `3` cũng không sao: heap chỉ quan tâm quan hệ **cha – con**, không quan tâm "anh em họ".

Hàm `show_heap_levels` dưới đây in heap theo tầng; tầng `t` chứa tối đa `2^t` phần tử.

In [10]:
def show_heap_levels(h):
    level, start = 0, 0
    while start < len(h):
        size = 2 ** level                        # tầng t có tối đa 2^t phần tử
        print(f"  tầng {level}: {h[start:start + size]}")
        start += size
        level += 1


def check_heap_property(h):
    # cha luôn <= con, tức h[(i-1)//2] <= h[i] với mọi i >= 1
    return all(h[(i - 1) // 2] <= h[i] for i in range(1, len(h)))


h = []
for x in [5, 1, 8, 3, 2]:
    heapq.heappush(h, x)
print("heap =", h)
show_heap_levels(h)
print("cha <= con ở mọi chỗ?", check_heap_property(h))
print("list có được sắp xếp không?", h == sorted(h))

heap = [1, 2, 8, 5, 3]
  tầng 0: [1]
  tầng 1: [2, 8]
  tầng 2: [5, 3]
cha <= con ở mọi chỗ? True
list có được sắp xếp không? False


### 6.3 Vì sao push / pop chỉ tốn `O(log m)`?

- **push**: đặt phần tử mới vào **cuối list** (lá cuối cùng của cây), rồi cho nó **"nổi lên"** (sift up): nếu nó nhỏ hơn cha thì đổi chỗ với cha, lặp lại. Mỗi lần đổi chỗ đi lên đúng 1 tầng.
- **pop**: lấy gốc ra, đưa phần tử cuối list lên gốc, rồi cho nó **"chìm xuống"** (sift down): đổi chỗ với con nhỏ hơn cho tới khi đúng vị trí. Mỗi lần đổi chỗ đi xuống 1 tầng.

Vậy số lần đổi chỗ tối đa = **số tầng** của cây. Tầng `t` chứa `2^t` phần tử, nên 16 tầng đã chứa được `2^16 - 1 = 65 535` phần tử. Với `m = 5·10^4` phần tử, cây chỉ cao 16 tầng → mỗi push / pop tốn tối đa khoảng 15 lần đổi chỗ. Số tầng ≈ `log2(m)` → `O(log m)`.

Cell dưới tự cài push có in từng lần "nổi lên", rồi so với `heapq.heappush`.

In [11]:
import math


def push_with_trace(h, x):
    h.append(x)                                  # 1) đặt ở cuối list
    i = len(h) - 1
    print(f"đặt {x} ở cuối: {h}")
    while i > 0:                                 # 2) nổi lên khi còn nhỏ hơn cha
        parent = (i - 1) // 2
        if h[i] < h[parent]:
            print(f"  {h[i]} < cha {h[parent]} -> đổi chỗ", end="")
            h[i], h[parent] = h[parent], h[i]
            print(f": {h}")
            i = parent
        else:
            print(f"  {h[i]} >= cha {h[parent]} -> dừng")
            break
    if i == 0:
        print("  đã lên tới gốc -> dừng")


h1 = [1, 2, 8, 5, 3]
h2 = list(h1)
push_with_trace(h1, 0)
heapq.heappush(h2, 0)
print("heapq.heappush cho cùng kết quả?", h1 == h2)

for m in [5, 1000, 5 * 10**4]:
    print(f"m = {m:>6} phần tử -> cây có {math.floor(math.log2(m)) + 1} tầng")

đặt 0 ở cuối: [1, 2, 8, 5, 3, 0]
  0 < cha 8 -> đổi chỗ: [1, 2, 0, 5, 3, 8]
  0 < cha 1 -> đổi chỗ: [0, 2, 1, 5, 3, 8]
  đã lên tới gốc -> dừng
heapq.heappush cho cùng kết quả? True
m =      5 phần tử -> cây có 3 tầng
m =   1000 phần tử -> cây có 10 tầng
m =  50000 phần tử -> cây có 16 tầng


### 6.4 Python chỉ có min-heap → làm **max-heap** bằng số âm

`heapq` chỉ có **min-heap** (gốc nhỏ nhất). Bài này còn cần **max-heap** (lấy nhanh số **lớn nhất** của nửa dưới). Mẹo kinh điển: **lưu số đối `-x` thay vì `x`**.

Vì sao được? Đổi dấu làm **đảo ngược thứ tự**: `a < b` khi và chỉ khi `-a > -b`. Số lớn nhất trong dữ liệu thật trở thành số âm **nhỏ nhất** → nằm ở gốc của min-heap.

Ví dụ: muốn max-heap chứa `5, 1, 8` → lưu `-5, -1, -8`. Min-heap cho gốc là `-8` → đổi dấu lại được `8` = số lớn nhất.

Quy tắc "vào âm, ra âm" cho hai heap của bài này:

| Việc cần làm | `high` (min-heap, lưu số thật) | `low` (max-heap, lưu số âm) |
|---|---|---|
| thêm `x` | `heapq.heappush(high, x)` | `heapq.heappush(low, -x)` |
| xem đỉnh | `high[0]` | `-low[0]` |
| lấy đỉnh ra | `heapq.heappop(high)` | `-heapq.heappop(low)` |

Chuyển đỉnh của `low` sang `high` viết gọn trong 1 dòng: `heapq.heappush(high, -heapq.heappop(low))`. Đọc từ trong ra ngoài: `heappop(low)` lấy số âm nhỏ nhất (ví dụ `-8`), dấu `-` biến nó thành `8` (số lớn nhất của `low`), rồi push `8` vào `high`. Chiều ngược lại: `heapq.heappush(low, -heapq.heappop(high))`.

In [12]:
low = []
for x in [5, 1, 8]:
    heapq.heappush(low, -x)                  # lưu số ÂM
print("low lưu thô (số âm):", low)
print("max của low =", -low[0])              # đổi dấu khi đọc

high = [10, 20]
heapq.heapify(high)
heapq.heappush(high, -heapq.heappop(low))    # chuyển max của low (8) sang high
print("sau khi chuyển: low (giá trị thật) =", sorted(-x for x in low), "| high =", sorted(high))
print("đỉnh low =", -low[0], "| đỉnh high =", high[0])

low lưu thô (số âm): [-8, -1, -5]
max của low = 8
sau khi chuyển: low (giá trị thật) = [1, 5] | high = [8, 10, 20]
đỉnh low = 5 | đỉnh high = 8


## 7. Bước đột phá — chia dữ liệu làm hai nửa

### 7.1 Median chỉ phụ thuộc vào hai "đỉnh"

Quay lại dữ liệu đã sắp xếp `[1, 3, 5, 8, 15]` và cắt nó làm hai nửa:

```text
     nửa dưới (low)            nửa trên (high)
     1    3    [5]      |      [8]    15
                ^                ^
     số LỚN NHẤT của low     số NHỎ NHẤT của high
```

- `n = 5` lẻ: ta quy ước `low` giữ nhiều hơn 1 phần tử → median = số lớn nhất của `low` = `5`.
- Thêm thử để có `[1, 3, 5, 8]` (`n = 4` chẵn): `low = [1, 3]`, `high = [5, 8]` → median = `(3 + 5) / 2 = 4.0`.

Ta **không cần** biết `1` đứng trước `3`, hay `15` đứng sau `8`! Chỉ cần đọc nhanh hai giá trị: **max của `low`** và **min của `high`**. Vậy:

- `low` = **max-heap** (lưu số âm) → `-low[0]` là số lớn nhất nửa dưới.
- `high` = **min-heap** → `high[0]` là số nhỏ nhất nửa trên.

Để công thức trên luôn đúng, hai heap phải luôn thỏa **2 invariant** (bất biến — điều luôn đúng sau mỗi lần `addNum`):

- **(I1) Thứ tự:** mọi phần tử của `low` ≤ mọi phần tử của `high` (tương đương: max của `low` ≤ min của `high`).
- **(I2) Kích thước:** `len(low) == len(high)` hoặc `len(low) == len(high) + 1`.

Khi đó:

```python
if len(low) > len(high):         # n lẻ
    median = -low[0]
else:                            # n chẵn
    median = (-low[0] + high[0]) / 2
```

(Cho `low` giữ phần dư khi `n` lẻ chỉ là **quy ước**. Có thể cho `high` giữ cũng được, miễn là `addNum` và `findMedian` dùng **cùng một** quy ước.)

Cell dưới kiểm chứng: cắt một list đã sắp xếp thành `low` (lấy `ceil(n/2)` phần tử đầu) và `high`, tính median từ hai đỉnh, so với định nghĩa trên 1000 list ngẫu nhiên.

In [13]:
import random


def median_from_halves(sorted_arr):
    n = len(sorted_arr)
    cut = (n + 1) // 2                       # low lấy ceil(n/2) phần tử nhỏ nhất
    low, high = sorted_arr[:cut], sorted_arr[cut:]
    if len(low) > len(high):                 # n lẻ
        return float(max(low))
    return (max(low) + min(high)) / 2        # n chẵn


for arr in [[1, 3, 5, 8, 15], [1, 3, 5, 8], [4], [2, 2, 2, 7]]:
    cut = (len(arr) + 1) // 2
    print(f"{str(arr):<17} low = {str(arr[:cut]):<11} high = {str(arr[cut:]):<9} -> median = {median_from_halves(arr)}")

random.seed(0)
for _ in range(1000):
    arr = sorted(random.randint(-50, 50) for _ in range(random.randint(1, 12)))
    assert median_from_halves(arr) == median_of_sorted(arr)
print("1000 list ngẫu nhiên: median tính từ 2 đỉnh == median theo định nghĩa")

[1, 3, 5, 8, 15]  low = [1, 3, 5]   high = [8, 15]   -> median = 5.0
[1, 3, 5, 8]      low = [1, 3]      high = [5, 8]    -> median = 4.0
[4]               low = [4]         high = []        -> median = 4.0
[2, 2, 2, 7]      low = [2, 2]      high = [2, 7]    -> median = 2.0
1000 list ngẫu nhiên: median tính từ 2 đỉnh == median theo định nghĩa


### 7.2 Thêm một số mới thì giữ 2 invariant thế nào?

Mỗi lần `addNum(num)` gồm đúng 2 bước:

**Bước 1 — đặt `num` vào đúng nửa (giữ I1).** So `num` với số lớn nhất của `low`:

- `low` rỗng, hoặc `num <= -low[0]` → `num` thuộc nửa dưới → push vào `low`.
- ngược lại (`num > -low[0]`) → `num` thuộc nửa trên → push vào `high`.

**Bước 2 — cân bằng kích thước (sửa I2).** Sau bước 1, một bên có thể bị dư:

- `len(low) > len(high) + 1` → chuyển **đỉnh của `low`** (số lớn nhất nửa dưới) sang `high`.
- `len(high) > len(low)` → chuyển **đỉnh của `high`** (số nhỏ nhất nửa trên) về `low`.

Ví dụ với `low = {1, 3, 5}`, `high = {8, 15}`:

- Thêm `6`: `6 > 5` → vào `high` = `{6, 8, 15}`. Kích thước 3 / 3 → không cần cân bằng. Median = `(5 + 6) / 2 = 5.5`. Kiểm tra: `[1, 3, 5, 6, 8, 15]` → `(5 + 6) / 2 = 5.5`. Đúng.
- Thêm `2` (thay vì `6`): `2 <= 5` → vào `low` = `{1, 2, 3, 5}`. Kích thước 4 / 2 → `low` dư → chuyển `5` sang `high`: `low = {1, 2, 3}`, `high = {5, 8, 15}`. Median = `(3 + 5) / 2 = 4.0`. Kiểm tra: `[1, 2, 3, 5, 8, 15]` → `4.0`. Đúng.

**Vì sao chuyển đỉnh không phá I1?** Đỉnh của `low` lớn hơn hoặc bằng mọi phần tử còn lại của `low`, và (theo I1) nhỏ hơn hoặc bằng mọi phần tử của `high`. Chuyển nó sang `high`, nó trở thành phần tử nhỏ nhất mới của `high` → thứ tự vẫn đúng. Chiều ngược lại tương tự.

**Vì sao mỗi lần chỉ cần chuyển tối đa 1 phần tử?** Trước khi thêm, hiệu `len(low) - len(high)` là `0` hoặc `1`. Bước 1 làm hiệu tăng hoặc giảm đúng 1 → hiệu chỉ có thể là `-1, 0, 1, 2`. Hiệu `2` → chuyển 1 phần tử thì về `0`; hiệu `-1` → chuyển 1 phần tử thì về `1`. Không bao giờ cần chuyển 2 lần.

## 8. Cách 3 — Hai heap (cách chính)

Ghép 2 bước ở mục 7.2 thành code. Vài điểm cần đọc kỹ:

- `self.low` lưu **số âm** (max-heap), `self.high` lưu **số thật** (min-heap). Cả hai tạo trong `__init__`.
- `if not self.low or num <= -self.low[0]:` — `not self.low` là True khi list rỗng. Phải đặt vế này **trước** `or`: Python đánh giá kiểu "đoản mạch" (short-circuit), vế trái đã True thì **không** đọc vế phải, nên không bao giờ chạm `self.low[0]` khi `low` rỗng.
- Bước cân bằng dùng `if / elif`: hai trường hợp loại trừ nhau, mỗi lần tối đa một lần chuyển (đã chứng minh ở 7.2).
- `findMedian` dùng `float(...)` cho trường hợp lẻ và `/` (không phải `//`) cho trường hợp chẵn.

Khi `low` rỗng thì `high` cũng rỗng (theo I2, `len(low) >= len(high)`), nên đưa số đầu tiên vào `low` là hợp lệ.

In [14]:
class MedianFinderTwoHeaps:
    def __init__(self):
        self.low = []    # max-heap (lưu số ÂM): nửa nhỏ, -low[0] = số lớn nhất nửa nhỏ
        self.high = []   # min-heap (lưu số thật): nửa lớn, high[0] = số nhỏ nhất nửa lớn

    def addNum(self, num: int) -> None:
        # Bước 1: đặt num vào đúng nửa (giữ I1)
        if not self.low or num <= -self.low[0]:
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)
        # Bước 2: cân bằng kích thước (giữ I2)
        if len(self.low) > len(self.high) + 1:
            heapq.heappush(self.high, -heapq.heappop(self.low))
        elif len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self) -> float:
        if len(self.low) > len(self.high):           # n lẻ
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2     # n chẵn


print("Example 1 :", show(run_commands(MedianFinderTwoHeaps, ops1, args1)))
print("5,15,1,3,8:", show(run_commands(MedianFinderTwoHeaps, ops2, args2)))

Example 1 : [null, null, null, 1.5, null, 2.0]
5,15,1,3,8: [null, null, 5.0, null, 10.0, null, 5.0, null, 4.0, null, 5.0]


### 8.1 Trace bảng trạng thái hai heap với chuỗi `5, 15, 1, 3, 8`

Tính tay trước. Cột `low` và `high` ghi **giá trị thật**, xếp tăng dần cho dễ đọc (bên trong heap thì không sắp xếp như vậy). "Kích thước" là `len(low) / len(high)` **ngay sau bước 1** (trước khi cân bằng).

| Bước | `num` | Bước 1: so với đỉnh `low` | Kích thước sau bước 1 | Bước 2: cân bằng | `low` | `high` | `findMedian()` |
|---:|---:|---|---|---|---|---|---:|
| 1 | 5 | `low` rỗng → vào `low` | 1 / 0 | không cần | `[5]` | `[]` | `5.0` |
| 2 | 15 | `15 > 5` → vào `high` | 1 / 1 | không cần | `[5]` | `[15]` | `10.0` |
| 3 | 1 | `1 <= 5` → vào `low` | 2 / 1 | không cần | `[1, 5]` | `[15]` | `5.0` |
| 4 | 3 | `3 <= 5` → vào `low` | 3 / 1 | `low` dư → chuyển `5` sang `high` | `[1, 3]` | `[5, 15]` | `4.0` |
| 5 | 8 | `8 > 3` → vào `high` | 2 / 3 | `high` nhiều hơn → chuyển `5` về `low` | `[1, 3, 5]` | `[8, 15]` | `5.0` |

Chuỗi này được chọn vì nó kích hoạt **cả hai** nhánh cân bằng (bước 4 và bước 5). Hàm `trace_two_heaps` dưới đây chạy đúng thuật toán và in cùng những cột đó, kèm theo list "thô" bên trong `low` (số âm).

In [15]:
def trace_two_heaps(stream):
    low, high = [], []
    print("Stream:", stream)
    for step, num in enumerate(stream, start=1):
        # Bước 1: đặt vào đúng nửa
        if not low:
            where = "low rỗng -> vào low"
            heapq.heappush(low, -num)
        elif num <= -low[0]:
            where = f"{num} <= đỉnh low {-low[0]} -> vào low"
            heapq.heappush(low, -num)
        else:
            where = f"{num} > đỉnh low {-low[0]} -> vào high"
            heapq.heappush(high, num)
        sizes = f"{len(low)}/{len(high)}"
        # Bước 2: cân bằng
        if len(low) > len(high) + 1:
            moved = -heapq.heappop(low)
            heapq.heappush(high, moved)
            fix = f"low dư -> chuyển {moved} sang high"
        elif len(high) > len(low):
            moved = heapq.heappop(high)
            heapq.heappush(low, -moved)
            fix = f"high nhiều hơn -> chuyển {moved} về low"
        else:
            fix = "không cần cân bằng"
        median = float(-low[0]) if len(low) > len(high) else (-low[0] + high[0]) / 2
        print(f"Bước {step}: thêm {num:<3}| {where} | kích thước {sizes} | {fix}")
        print(f"   low = {sorted(-x for x in low)} (thô {low}) | high = {sorted(high)} | median = {median}")
    return median


last = trace_two_heaps([5, 15, 1, 3, 8])
assert last == 5.0

Stream: [5, 15, 1, 3, 8]
Bước 1: thêm 5  | low rỗng -> vào low | kích thước 1/0 | không cần cân bằng
   low = [5] (thô [-5]) | high = [] | median = 5.0
Bước 2: thêm 15 | 15 > đỉnh low 5 -> vào high | kích thước 1/1 | không cần cân bằng
   low = [5] (thô [-5]) | high = [15] | median = 10.0
Bước 3: thêm 1  | 1 <= đỉnh low 5 -> vào low | kích thước 2/1 | không cần cân bằng
   low = [1, 5] (thô [-5, -1]) | high = [15] | median = 5.0
Bước 4: thêm 3  | 3 <= đỉnh low 5 -> vào low | kích thước 3/1 | low dư -> chuyển 5 sang high
   low = [1, 3] (thô [-3, -1]) | high = [5, 15] | median = 4.0
Bước 5: thêm 8  | 8 > đỉnh low 3 -> vào high | kích thước 2/3 | high nhiều hơn -> chuyển 5 về low
   low = [1, 3, 5] (thô [-5, -1, -3]) | high = [8, 15] | median = 5.0


Output khớp bảng tính tay. Những điểm cần để ý:

- **Bước 4** kích hoạt nhánh "`low` dư": sau khi thêm `3`, `low` có 3 phần tử còn `high` chỉ có 1 → chuyển số lớn nhất của `low` (`5`) sang `high`.
- **Bước 5** kích hoạt nhánh ngược lại: `8` được so với đỉnh `low` lúc đó là **`3`** (không phải `5`, vì `5` đã sang `high`), nên `8` vào `high` → `high` có 3, `low` có 2 → chuyển số nhỏ nhất của `high` (lại là `5`) về `low`.
- Số `5` "đi qua đi lại" giữa hai heap — hoàn toàn bình thường. Invariant chỉ cần đúng **khi `addNum` kết thúc**, không cần đúng ở giữa chừng.
- List thô của `low` chứa số âm và không sắp xếp (ví dụ `[-5, -1, -3]`); chỉ phần tử đầu `low[0]` là số âm nhỏ nhất, tức số thật lớn nhất.
- Mỗi bước có **tối đa một** lần chuyển, đúng như lập luận ở 7.2.

### 8.2 Vì sao cách 3 luôn đúng?

Ta chứng minh 2 invariant đúng sau **mọi** lần `addNum` (lập luận quy nạp: đúng lúc đầu, và nếu đúng trước một lần thêm thì vẫn đúng sau lần thêm đó).

**Lúc đầu:** hai heap rỗng → I1 và I2 hiển nhiên đúng.

**Giả sử trước khi thêm `num`, I1 và I2 đúng.** Gọi `L` = số lớn nhất của `low`.

1. *Bước 1 giữ I1.*
   - Nếu `num <= L` → `num` vào `low`. Mọi phần tử cũ của `low` ≤ `L` ≤ mọi phần tử của `high`, và `num <= L` ≤ mọi phần tử của `high` → I1 vẫn đúng.
   - Nếu `num > L` → `num` vào `high`. Mọi phần tử của `low` ≤ `L` < `num` → I1 vẫn đúng.
   - Nếu `low` rỗng thì `high` cũng rỗng (theo I2) → đưa `num` vào `low` không phá gì.
2. *Bước 2 sửa I2 mà không phá I1.* Sau bước 1, hiệu `d = len(low) - len(high)` thuộc `{-1, 0, 1, 2}`.
   - `d = 2`: chuyển số lớn nhất của `low` sang `high`. Số này ≥ mọi phần tử còn lại của `low` và ≤ mọi phần tử của `high` → I1 giữ nguyên; `d` về `0`.
   - `d = -1`: chuyển số nhỏ nhất của `high` về `low`, lập luận đối xứng; `d` về `1`.
   - `d = 0` hoặc `1`: không làm gì.

Vậy sau `addNum`, I1 và I2 lại đúng.

**Vì sao `findMedian` trả đúng median?** Theo I1, nếu xếp `low` tăng dần rồi nối tiếp `high` tăng dần thì ta được **toàn bộ dữ liệu đã sắp xếp**. Theo I2, `low` chứa đúng `ceil(n/2)` phần tử nhỏ nhất.

- `n` lẻ: `len(low) = (n + 1) / 2`, nên số lớn nhất của `low` là phần tử thứ `(n + 1) / 2` (đếm từ 1) của dãy sắp xếp — chính là phần tử giữa. Ví dụ `n = 5`: `low` có 3 phần tử, max của `low` là phần tử thứ 3.
- `n` chẵn: `len(low) = len(high) = n / 2`, hai phần tử giữa là phần tử thứ `n/2` (= max của `low`) và thứ `n/2 + 1` (= min của `high`).

Cell dưới kiểm chứng bằng máy: sau **mỗi** lần `addNum` trên hàng nghìn lần thêm ngẫu nhiên, ta kiểm tra I1, I2 và so median với định nghĩa.

In [16]:
def check_invariants(mf):
    low_vals = [-x for x in mf.low]
    high_vals = list(mf.high)
    if high_vals:
        assert max(low_vals) <= min(high_vals), "I1 bị vỡ"
        assert mf.high[0] == min(high_vals), "đỉnh high phải là min"
    assert len(low_vals) - len(high_vals) in (0, 1), "I2 bị vỡ"
    assert -mf.low[0] == max(low_vals), "đỉnh low phải là max"


random.seed(1)
total_adds = 0
for _ in range(300):
    mf = MedianFinderTwoHeaps()
    data = []
    for _ in range(random.randint(1, 30)):
        x = random.randint(-20, 20)
        mf.addNum(x)
        data.append(x)
        total_adds += 1
        check_invariants(mf)
        assert mf.findMedian() == median_of_sorted(sorted(data))
print(f"Đã kiểm tra sau {total_adds} lần addNum: I1, I2 luôn đúng và median luôn khớp định nghĩa.")

Đã kiểm tra sau 4608 lần addNum: I1, I2 luôn đúng và median luôn khớp định nghĩa.


### 8.3 Độ phức tạp cách 3

`n` = số phần tử đã thêm.

- `addNum`: 1 lần push + (có thể) 1 lần pop + 1 lần push → tối đa 3 thao tác heap, mỗi thao tác `O(log n)` → **`O(log n)`**. Với `n = 5·10^4`, `log2 n ≈ 16` → mỗi lần thêm chỉ vài chục bước.
- `findMedian`: chỉ đọc `low[0]` và `high[0]` → **`O(1)`**.
- Bộ nhớ: **`O(n)`** — mọi số đều nằm trong đúng 1 trong 2 heap.

Tổng cho `q` lời gọi: `O(q log q)`. Với `q = 5·10^4` chỉ cỡ một triệu bước — rất thoải mái so với hàng tỷ bước của cách 1.

So với cách 2: ta đã đổi `O(n)` thành `O(log n)` cho `addNum` bằng cách **từ bỏ** việc giữ thứ tự toàn bộ, chỉ giữ đúng thông tin cần cho median.

## 9. Code nộp LeetCode

Khi nộp bài:

- Giữ **nguyên** tên class `MedianFinder` và tên method `addNum`, `findMedian`: LeetCode gọi theo tên (giống hệt hàm `run_commands` của ta).
- LeetCode tạo **object mới** cho mỗi test case (`obj = MedianFinder()`), nên mọi trạng thái phải được khởi tạo trong `__init__` bằng `self.`.
- `import heapq`: môi trường LeetCode thường đã import sẵn, nhưng cứ ghi rõ để code chạy được ở mọi nơi.
- Code nộp bài nên gọn, không `print` debug.

In [17]:
import heapq


class MedianFinder:

    def __init__(self):
        self.low = []    # max-heap (lưu số âm): nửa nhỏ, -low[0] là số lớn nhất của nửa nhỏ
        self.high = []   # min-heap: nửa lớn, high[0] là số nhỏ nhất của nửa lớn

    def addNum(self, num: int) -> None:
        if not self.low or num <= -self.low[0]:
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)

        if len(self.low) > len(self.high) + 1:
            heapq.heappush(self.high, -heapq.heappop(self.low))
        elif len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


# Your MedianFinder object will be instantiated and called as such:
# obj = MedianFinder()
# obj.addNum(num)
# param_2 = obj.findMedian()

### 9.1 Bộ test cho `MedianFinder`

Ta kiểm tra các nhóm:

1. **Example 1** của đề, chạy đúng 2 list `ops`, `args` như LeetCode.
2. **Một phần tử** (biên nhỏ nhất theo constraints: phải có ít nhất 1 phần tử trước khi hỏi).
3. **Hỏi nhiều lần liên tiếp**: `findMedian` không được làm thay đổi trạng thái.
4. **Số âm**: dễ lộ lỗi `//` và lỗi quên đổi dấu.
5. **Toàn số trùng nhau**.
6. **Tăng dần / giảm dần**: dễ lộ lỗi quên cân bằng (mọi số dồn về một heap).
7. **Biên giá trị** `±10^5`.
8. Chuỗi `5, 15, 1, 3, 8` đã trace.

Hàm `same_outputs` so hai list output, chấp nhận sai số `10^-5` như đề nói. Hàm `expected_from` ghép output mong đợi từ danh sách median, để ta chỉ cần viết các median mong đợi.

In [18]:
def same_outputs(got, expected, tol=1e-5):
    # So 2 list output kiểu LeetCode, chấp nhận sai số tol như đề
    if len(got) != len(expected):
        return False
    for g, e in zip(got, expected):
        if g is None or e is None:
            if g is not e:
                return False
        elif abs(g - e) > tol:
            return False
    return True


def expected_from(tokens, medians):
    # None cho constructor và mỗi addNum; lần lượt các median cho mỗi "?"
    it = iter(medians)
    return [None] + [next(it) if t == "?" else None for t in tokens]


def check_cases(cls, cases, verbose=False):
    for name, tokens, medians in cases:
        ops, args = build_commands(tokens)
        got = run_commands(cls, ops, args)
        assert same_outputs(got, expected_from(tokens, medians)), (cls.__name__, name, got)
        assert all(isinstance(x, float) for x in got if x is not None), "findMedian phải trả float"
        if verbose:
            print(f"  PASS  {name:<32} medians = {[x for x in got if x is not None]}")
    print(f"{cls.__name__}: {len(cases)} test PASS")


# 1) Example 1 đúng nguyên văn input của LeetCode
ops_ex = ["MedianFinder", "addNum", "addNum", "findMedian", "addNum", "findMedian"]
args_ex = [[], [1], [2], [], [3], []]
out = run_commands(MedianFinder, ops_ex, args_ex)
assert out == [None, None, None, 1.5, None, 2.0]
print("Example 1:", show(out))

basic_cases = [
    ("Example 1", [1, 2, "?", 3, "?"], [1.5, 2.0]),
    ("một phần tử", [7, "?"], [7.0]),
    ("hỏi liên tiếp, state không đổi", [5, "?", "?", "?"], [5.0, 5.0, 5.0]),
    ("số âm", [-1, "?", -2, "?", -3, "?"], [-1.0, -1.5, -2.0]),
    ("toàn số trùng", [2, 2, 2, 2, "?", 2, "?"], [2.0, 2.0]),
    ("tăng dần 1..6", [1, "?", 2, "?", 3, "?", 4, "?", 5, "?", 6, "?"], [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]),
    ("giảm dần 6..1", [6, "?", 5, "?", 4, "?", 3, "?", 2, "?", 1, "?"], [6.0, 5.5, 5.0, 4.5, 4.0, 3.5]),
    ("biên ±10^5", [10**5, -10**5, "?", 10**5, "?"], [0.0, 100000.0]),
    ("chuỗi 5,15,1,3,8", [5, "?", 15, "?", 1, "?", 3, "?", 8, "?"], [5.0, 10.0, 5.0, 4.0, 5.0]),
]
check_cases(MedianFinder, basic_cases, verbose=True)

Example 1: [null, null, null, 1.5, null, 2.0]
  PASS  Example 1                        medians = [1.5, 2.0]
  PASS  một phần tử                      medians = [7.0]
  PASS  hỏi liên tiếp, state không đổi   medians = [5.0, 5.0, 5.0]
  PASS  số âm                            medians = [-1.0, -1.5, -2.0]
  PASS  toàn số trùng                    medians = [2.0, 2.0]
  PASS  tăng dần 1..6                    medians = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
  PASS  giảm dần 6..1                    medians = [6.0, 5.5, 5.0, 4.5, 4.0, 3.5]
  PASS  biên ±10^5                       medians = [0.0, 100000.0]
  PASS  chuỗi 5,15,1,3,8                 medians = [5.0, 10.0, 5.0, 4.0, 5.0]
MedianFinder: 9 test PASS


### 9.2 Bản gọn hơn: cho số mới "đi vòng qua" `low` bằng `heappushpop`

`heapq.heappushpop(h, x)` = push `x` rồi pop phần tử nhỏ nhất, làm trong **một** thao tác (nhanh hơn gọi riêng hai lệnh). Nếu `x <= h[0]` (hoặc heap rỗng) nó trả lại `x` ngay mà không đụng vào heap.

Ý tưởng của bản gọn: không cần `if / else` để chọn nửa, mà **luôn** làm 2 bước:

1. Đưa `num` vào `low` rồi lấy ra số lớn nhất của (`low` + `num`), đẩy số đó sang `high`. Số bị đẩy sang là số lớn nhất của nửa dưới → I1 được giữ **tự động**.
2. Nếu `high` nhiều hơn `low` → trả số nhỏ nhất của `high` về `low`.

Về kích thước: trước khi thêm, `len(low) - len(high)` là `0` hoặc `1`. Bước 1 giữ nguyên `len(low)` và tăng `len(high)` thêm 1 → hiệu là `-1` hoặc `0`. Bước 2 sửa `-1` thành `1`. Không cần nhánh "`low` dư".

Đổi lại: bản gọn **luôn** tốn 2–3 thao tác heap kể cả khi không cần, và khó "nhìn ra" lý do hơn bản `if / else`. Cả hai đều `O(log n)` cho `addNum`, `O(1)` cho `findMedian`.

In [19]:
h = [3, 5, 9]
print(heapq.heappushpop(h, 1), h)    # 1 <= h[0] -> trả 1 ngay, heap giữ nguyên
print(heapq.heappushpop(h, 7), h)    # push 7 rồi pop 3 -> trả 3, heap = [5, 7, 9]


class MedianFinderPushPop:
    def __init__(self):
        self.low = []    # max-heap (số âm)
        self.high = []   # min-heap

    def addNum(self, num: int) -> None:
        # 1) num "đi qua" low: low nhận num rồi nhả ra số lớn nhất -> đẩy sang high
        biggest_low = -heapq.heappushpop(self.low, -num)
        heapq.heappush(self.high, biggest_low)
        # 2) high không được nhiều hơn low -> trả số nhỏ nhất của high về low
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


check_cases(MedianFinderPushPop, basic_cases)

1 [3, 5, 9]
3 [5, 7, 9]
MedianFinderPushPop: 9 test PASS


## 10. Follow-up 1 — Mọi số nằm trong `[0, 100]`: Cách 4, mảng đếm

Khi miền giá trị chỉ có **101 khả năng** (`0, 1, ..., 100`), dù stream có `5·10^4` số thì cũng chỉ có tối đa 101 giá trị **khác nhau**. Thay vì lưu từng số, ta chỉ cần biết **mỗi giá trị xuất hiện bao nhiêu lần**.

### 10.1 Mảng đếm (counting array)

- `count = [0] * 101` — ô `count[v]` = số lần giá trị `v` đã xuất hiện.
- `addNum(num)`: `count[num] += 1` → `O(1)`. Giá trị `num` dùng làm **chỉ số** luôn, không cần tìm kiếm gì.

Ví dụ stream `[2, 7, 2, 5, 9, 2]` → `count[2] = 3`, `count[5] = 1`, `count[7] = 1`, `count[9] = 1`, các ô khác bằng 0.

Mảng đếm thật ra là "list đã sắp xếp được nén lại": đọc từ ô `0` tới ô `100`, viết giá trị `v` ra `count[v]` lần, ta được đúng `[2, 2, 2, 5, 7, 9]`.

### 10.2 Tìm phần tử nhỏ thứ `k` bằng đếm tích lũy

Muốn lấy median, ta cần **phần tử nhỏ thứ `k`** (đếm từ 1) của dữ liệu. Cách làm: đi từ `v = 0` tới `100`, cộng dồn `cum += count[v]`. **Ô đầu tiên làm `cum >= k`** chính là đáp án.

Vì sao? Sau khi cộng ô `v`, `cum` = **số phần tử ≤ `v`**. Ô đầu tiên có `cum >= k` nghĩa là: số phần tử ≤ `v - 1` còn **ít hơn** `k`, nhưng số phần tử ≤ `v` đã **đủ** `k` → phần tử thứ `k` phải bằng `v`.

Với stream `[2, 7, 2, 5, 9, 2]` (`n = 6`, chẵn), hai phần tử giữa là phần tử thứ `k = 3` và `k = 4`:

| `v` | `count[v]` | `cum` (số phần tử ≤ `v`) | Tìm `k = 3` | Tìm `k = 4` |
|---:|---:|---:|---|---|
| 0 | 0 | 0 | | |
| 1 | 0 | 0 | | |
| 2 | 3 | 3 | **`3 >= 3` → đáp án `2`** | `3 < 4`, đi tiếp |
| 3 | 0 | 3 | | `3 < 4`, đi tiếp |
| 4 | 0 | 3 | | `3 < 4`, đi tiếp |
| 5 | 1 | 4 | | **`4 >= 4` → đáp án `5`** |

Median = `(2 + 5) / 2 = 3.5`. Kiểm tra: `[2, 2, 2, 5, 7, 9]` → hai phần tử giữa là `2` và `5` → `3.5`. Đúng. Để ý hai phần tử giữa có thể nằm ở **hai ô khác nhau** của mảng đếm.

In [20]:
stream = [2, 7, 2, 5, 9, 2]
count = [0] * 101                      # ô v đếm số lần giá trị v xuất hiện, v = 0..100
for num in stream:
    count[num] += 1
print("các ô khác 0 (giá trị, số lần):", [(v, count[v]) for v in range(101) if count[v] > 0])

# "giải nén" mảng đếm -> chính là list đã sắp xếp
unpacked = [v for v in range(101) for _ in range(count[v])]
print("giải nén:", unpacked, "| bằng sorted(stream)?", unpacked == sorted(stream))


def kth_from_counts(count, k, verbose=False):
    # Phần tử nhỏ thứ k (k tính từ 1) dựa trên mảng đếm
    cum = 0
    for value in range(len(count)):
        cum += count[value]                     # cum = số phần tử <= value
        if verbose:
            note = "-> cum >= k, dừng" if cum >= k else ""
            print(f"   v = {value}: count = {count[value]}, cum = {cum} {note}")
        if cum >= k:
            return value


print("Tìm phần tử nhỏ thứ k = 4:")
print("=> kết quả:", kth_from_counts(count, 4, verbose=True))
a3, a4 = kth_from_counts(count, 3), kth_from_counts(count, 4)
print(f"thứ 3 = {a3}, thứ 4 = {a4} -> median (n = 6) = ({a3} + {a4}) / 2 = {(a3 + a4) / 2}")

các ô khác 0 (giá trị, số lần): [(2, 3), (5, 1), (7, 1), (9, 1)]
giải nén: [2, 2, 2, 5, 7, 9] | bằng sorted(stream)? True
Tìm phần tử nhỏ thứ k = 4:
   v = 0: count = 0, cum = 0 
   v = 1: count = 0, cum = 0 
   v = 2: count = 3, cum = 3 
   v = 3: count = 0, cum = 3 
   v = 4: count = 0, cum = 3 
   v = 5: count = 1, cum = 4 -> cum >= k, dừng
=> kết quả: 5
thứ 3 = 2, thứ 4 = 5 -> median (n = 6) = (2 + 5) / 2 = 3.5


### 10.3 Class cho follow-up 1 và độ phức tạp

Chuyển "chỉ số giữa" (đếm từ 0) sang "phần tử thứ `k`" (đếm từ 1) bằng cách cộng 1:

- `n` lẻ: phần tử giữa ở chỉ số `n // 2` → `k = n // 2 + 1`. Ví dụ `n = 5` → `k = 3`.
- `n` chẵn: hai chỉ số `n // 2 - 1` và `n // 2` → `k = n // 2` và `k = n // 2 + 1`. Ví dụ `n = 6` → `k = 3` và `4`.

Class lưu thêm `self.n` (tổng số phần tử) để khỏi phải cộng cả mảng đếm mỗi lần hỏi.

**Độ phức tạp cách 4:**

- `addNum`: **`O(1)`** — chỉ tăng một ô.
- `findMedian`: duyệt tối đa 101 ô (trường hợp chẵn duyệt 2 lần) → **`O(101)` = `O(1)`**, vì 101 là hằng số, không phụ thuộc `n`.
- Bộ nhớ: **`O(101)` = `O(1)`** — chỉ 101 ô dù stream có hàng triệu số. So với hai heap (bộ nhớ `O(n)`), đây là khác biệt lớn nhất.

(Nâng cao: có thể giữ thêm một "con trỏ" tới ô chứa median và dịch con trỏ mỗi lần thêm để `findMedian` không phải duyệt từ đầu. Với chỉ 101 ô thì không cần thiết.)

In [21]:
class MedianFinderCounting:
    # Chỉ dùng khi MỌI số nằm trong [0, 100]
    def __init__(self):
        self.count = [0] * 101
        self.n = 0

    def addNum(self, num: int) -> None:
        self.count[num] += 1                    # O(1)
        self.n += 1

    def _kth(self, k):
        # phần tử nhỏ thứ k (k tính từ 1), duyệt tối đa 101 ô
        cum = 0
        for value in range(101):
            cum += self.count[value]
            if cum >= k:
                return value

    def findMedian(self) -> float:
        n = self.n
        if n % 2 == 1:
            return float(self._kth(n // 2 + 1))
        return (self._kth(n // 2) + self._kth(n // 2 + 1)) / 2


ops_c, args_c = build_commands([2, 7, 2, 5, 9, 2, "?"])
print("[2, 7, 2, 5, 9, 2] ->", show(run_commands(MedianFinderCounting, ops_c, args_c)))

in_range_cases = [c for c in basic_cases if all(t == "?" or 0 <= t <= 100 for t in c[1])]
counting_cases = in_range_cases + [
    ("hai middle ở 2 ô khác nhau", [2, 7, 2, 5, 9, 2, "?"], [3.5]),
    ("biên 0 và 100", [0, "?", 100, "?"], [0.0, 50.0]),
    ("toàn 100", [100, 100, 100, "?"], [100.0]),
]
check_cases(MedianFinderCounting, counting_cases)

[2, 7, 2, 5, 9, 2] -> [null, null, null, null, null, null, null, 3.5]
MedianFinderCounting: 10 test PASS


## 11. Follow-up 2 — 99% số nằm trong `[0, 100]`

### 11.1 Ý tưởng: mảng đếm + hai bộ đếm ngoài khoảng

Vẫn dùng mảng đếm 101 ô cho các số trong `[0, 100]`, và thêm **hai bộ đếm**:

- `n_below` = số phần tử **nhỏ hơn 0**;
- `n_above` = số phần tử **lớn hơn 100**.

Nếu xếp toàn bộ dữ liệu tăng dần, nó gồm 3 vùng nối tiếp nhau: `[ vùng < 0 ][ vùng 0..100 ][ vùng > 100 ]`. Muốn tìm phần tử thứ `k`:

- `k <= n_below` → nó nằm trong vùng `< 0`;
- `n_below < k <= n_below + n_in` (với `n_in` = số phần tử trong `[0, 100]`) → nó nằm trong mảng đếm, ở vị trí thứ `k - n_below` của vùng giữa;
- còn lại → nó nằm trong vùng `> 100`.

**Vì sao gần như luôn rơi vào mảng đếm?** Nếu ít nhất 99% số nằm trong `[0, 100]` thì `n_below` và `n_above` đều ≤ 1% của `n`. Median nằm ở vị trí khoảng `n / 2`. Ví dụ `n = 10 000`: `n_below <= 100`, `n_above <= 100`, nên mọi vị trí từ 101 tới 9 900 đều thuộc mảng đếm; median ở vị trí 5 000 và 5 001 nằm gọn trong đó. Vậy hai bộ đếm là đủ để **định vị** median, và `findMedian` vẫn chỉ duyệt 101 ô.

### 11.2 Nhưng chỉ đếm thôi thì có lỗ hổng

"99%" là thống kê của **cả** stream. Ở những thời điểm đầu (mới vài số), có thể toàn số ngoài khoảng. Ví dụ stream bắt đầu bằng `-5, -3, 200` → median = `-3` nằm **ngoài** `[0, 100]`. Nếu ta chỉ **đếm** số âm mà không **lưu giá trị** của chúng thì không thể trả lời "phần tử thứ 2 là bao nhiêu". Cell dưới cài bản "chỉ đếm" để thấy rõ: nó trả lời được ở tình huống thường gặp nhưng trả `None` ở tình huống hiếm.

In [22]:
import statistics


class MedianFinderCountersOnly:
    # Bản "chỉ đếm": đếm số ngoài khoảng nhưng KHÔNG lưu giá trị của chúng
    def __init__(self):
        self.count = [0] * 101
        self.n_in = 0
        self.n_below = 0          # số phần tử < 0
        self.n_above = 0          # số phần tử > 100

    def addNum(self, num: int) -> None:
        if num < 0:
            self.n_below += 1
        elif num > 100:
            self.n_above += 1
        else:
            self.count[num] += 1
            self.n_in += 1

    def _kth(self, k):
        if k <= self.n_below:
            return None                        # nằm trong vùng < 0 nhưng ta không lưu giá trị!
        k -= self.n_below
        if k > self.n_in:
            return None                        # nằm trong vùng > 100: cũng không biết giá trị
        cum = 0
        for value in range(101):
            cum += self.count[value]
            if cum >= k:
                return value

    def findMedian(self):
        n = self.n_below + self.n_in + self.n_above
        if n % 2 == 1:
            m = self._kth(n // 2 + 1)
            return None if m is None else float(m)
        a, b = self._kth(n // 2), self._kth(n // 2 + 1)
        return None if a is None or b is None else (a + b) / 2


random.seed(99)
mostly = [random.randint(0, 100) for _ in range(99)] + [100000, -7]
random.shuffle(mostly)
mf = MedianFinderCountersOnly()
for x in mostly:
    mf.addNum(x)
print(f"101 số (99 trong khoảng, n_below = {mf.n_below}, n_above = {mf.n_above}):",
      "median =", mf.findMedian(), "| đúng:", float(statistics.median(mostly)))

mf2 = MedianFinderCountersOnly()
for x in [-5, -3, 200]:
    mf2.addNum(x)
print("stream -5, -3, 200: median =", mf2.findMedian(), "| đúng:", float(statistics.median([-5, -3, 200])))

101 số (99 trong khoảng, n_below = 1, n_above = 1): median = 49.0 | đúng: 49.0
stream -5, -3, 200: median = None | đúng: -3.0


### 11.3 Cách 5 — Mảng đếm + cất giá trị ngoài khoảng vào 2 list đã sắp xếp

Sửa lỗ hổng: vẫn đếm như trên, nhưng **lưu luôn** các số ngoài khoảng vào hai list `below` (các số `< 0`) và `above` (các số `> 100`), giữ sắp xếp bằng `bisect.insort` (kỹ thuật của cách 2). Khi đó `len(below)` và `len(above)` chính là hai bộ đếm `n_below`, `n_above`.

- Nhờ hai list đã sắp xếp, phần tử thứ `k` trong vùng ngoài khoảng đọc được ngay: `below[k - 1]` hoặc `above[k - 1]`.
- Vì chỉ khoảng 1% số nằm ngoài khoảng, hai list này **nhỏ**, nên `insort` (`O(m)` với `m` = số phần tử ngoài khoảng) rất rẻ.

Hàm `_kth(k)` xét lần lượt 3 vùng: `below` → mảng đếm → `above`.

**Độ phức tạp cách 5** (`m` = số phần tử ngoài `[0, 100]`, khoảng 1% của `n`):

- `addNum`: `O(1)` với số trong khoảng; `O(m)` với số ngoài khoảng (hiếm).
- `findMedian`: tối đa 101 ô khi median nằm trong khoảng (gần như luôn vậy); `O(1)` khi median rơi vào `below` / `above` (đọc theo chỉ số).
- Bộ nhớ: `O(101 + m)` thay vì `O(n)`.

Cách 5 đúng với **mọi** input (không chỉ khi 99% trong khoảng), nên ta còn đưa nó vào test chéo với số bất kỳ ở mục 12.

In [23]:
class MedianFinderCounting99:
    def __init__(self):
        self.count = [0] * 101    # đếm cho các số trong [0, 100]
        self.n_in = 0             # số phần tử trong [0, 100]
        self.below = []           # các số < 0, giữ sắp xếp (len = bộ đếm n_below)
        self.above = []           # các số > 100, giữ sắp xếp (len = bộ đếm n_above)

    def addNum(self, num: int) -> None:
        if num < 0:
            bisect.insort(self.below, num)
        elif num > 100:
            bisect.insort(self.above, num)
        else:
            self.count[num] += 1
            self.n_in += 1

    def _kth(self, k):
        # phần tử nhỏ thứ k (k tính từ 1) trên toàn bộ dữ liệu: below -> mảng đếm -> above
        if k <= len(self.below):
            return self.below[k - 1]
        k -= len(self.below)
        if k <= self.n_in:
            cum = 0
            for value in range(101):
                cum += self.count[value]
                if cum >= k:
                    return value
        k -= self.n_in
        return self.above[k - 1]

    def findMedian(self) -> float:
        n = len(self.below) + self.n_in + len(self.above)
        if n % 2 == 1:
            return float(self._kth(n // 2 + 1))
        return (self._kth(n // 2) + self._kth(n // 2 + 1)) / 2


outside_cases = [
    ("median nằm trong vùng < 0", [-5, -3, 200, "?"], [-3.0]),
    ("median ở below rồi giữa 2 vùng", [-5, "?", 200, "?"], [-5.0, 97.5]),
    ("2 middle: below và mảng đếm", [-1, 0, "?"], [-0.5]),
    ("2 middle: mảng đếm và above", [100, 101, "?"], [100.5]),
    ("toàn số > 100", [101, 102, "?", 500, "?"], [101.5, 102.0]),
]
check_cases(MedianFinderCounting99, basic_cases + outside_cases)

mf3 = MedianFinderCounting99()
for x in mostly:                       # 101 số ở cell trước: 99 trong khoảng + 100000 + -7
    mf3.addNum(x)
print("101 số ở cell trước -> median =", mf3.findMedian(), "| below =", mf3.below, "| above =", mf3.above)

MedianFinderCounting99: 14 test PASS
101 số ở cell trước -> median = 49.0 | below = [-7] | above = [100000]


## 12. Test đối chiếu chéo trên chuỗi lệnh ngẫu nhiên

Ta sinh hàng trăm **chuỗi lệnh ngẫu nhiên** kiểu LeetCode, chạy tất cả các cách trên cùng chuỗi lệnh và so với một **oracle** (đáp án chắc chắn đúng): giữ list thô và mỗi lần hỏi dùng `statistics.median` của thư viện chuẩn.

- `random.seed(295)` cố định để lần chạy nào cũng ra cùng dữ liệu → nếu có lỗi thì tái hiện được.
- `random_tokens` đảm bảo **không bao giờ** gọi `findMedian` khi chưa có phần tử nào (đúng constraint của đề).
- Có 5 kịch bản: số bất kỳ trong `[-10^5, 10^5]`; nhiều số trùng trong `[-5, 5]`; mọi số trong `[0, 100]` (thêm cách 4); 99% trong `[0, 100]`; một nửa ngoài `[0, 100]` (ép median rơi ra ngoài để thử nhánh `below` / `above` của cách 5).
- Cuối cùng là test cỡ lớn đúng giới hạn `5·10^4` lời gọi. Ở cỡ này oracle (sort mỗi lần) quá chậm, nên ta dùng cách 2 (đã khớp oracle ở test nhỏ) làm mốc.

In [24]:
import random
import statistics


def oracle_outputs(tokens):
    # Đáp án chắc chắn đúng: giữ list thô, mỗi lần hỏi dùng statistics.median
    data, out = [], [None]
    for t in tokens:
        if t == "?":
            out.append(float(statistics.median(data)))
        else:
            data.append(t)
            out.append(None)
    return out


def random_tokens(num_calls, make_value, p_add=0.6):
    tokens, size = [], 0
    for _ in range(num_calls):
        if size == 0 or random.random() < p_add:     # chưa có phần tử thì bắt buộc addNum
            tokens.append(make_value())
            size += 1
        else:
            tokens.append("?")
    return tokens


def outlier():
    return random.choice([random.randint(-10**5, -1), random.randint(101, 10**5)])


general = {
    "1_sort_each_time": MedianFinderSortEachTime,
    "2_insort": MedianFinderInsort,
    "3_two_heaps": MedianFinderTwoHeaps,
    "3b_pushpop": MedianFinderPushPop,
    "5_counting_99": MedianFinderCounting99,
    "MedianFinder (nộp bài)": MedianFinder,
}
with_counting = {**general, "4_counting": MedianFinderCounting}

scenarios = [
    ("số bất kỳ trong [-10^5, 10^5]", lambda: random.randint(-10**5, 10**5), general),
    ("nhiều số trùng trong [-5, 5]", lambda: random.randint(-5, 5), general),
    ("mọi số trong [0, 100]", lambda: random.randint(0, 100), with_counting),
    ("99% trong [0, 100]", lambda: random.randint(0, 100) if random.random() < 0.99 else outlier(), general),
    ("một nửa ngoài [0, 100]", lambda: random.randint(0, 100) if random.random() < 0.5 else outlier(), general),
]

random.seed(295)
for title, make_value, methods in scenarios:
    for _ in range(300):
        tokens = random_tokens(random.randint(1, 40), make_value)
        ops, args = build_commands(tokens)
        expected = oracle_outputs(tokens)
        for name, cls in methods.items():
            got = run_commands(cls, ops, args)
            assert same_outputs(got, expected), (title, name, tokens, got)
    print(f"{title:<32}: {len(methods)} cách x 300 chuỗi lệnh -> khớp oracle")

# Test cỡ lớn: đúng 5·10^4 lời gọi
random.seed(2950)
for title, make_value, methods in [
    ("cỡ lớn, số bất kỳ", lambda: random.randint(-10**5, 10**5), [MedianFinder, MedianFinderPushPop, MedianFinderCounting99]),
    ("cỡ lớn, mọi số trong [0, 100]", lambda: random.randint(0, 100), [MedianFinder, MedianFinderCounting, MedianFinderCounting99]),
]:
    tokens = random_tokens(5 * 10**4, make_value)
    ops, args = build_commands(tokens)
    reference = run_commands(MedianFinderInsort, ops, args)
    for cls in methods:
        assert same_outputs(run_commands(cls, ops, args), reference), (title, cls.__name__)
    print(f"{title:<32}: {len(tokens)} lời gọi, {tokens.count('?')} lần findMedian -> khớp cách 2")

số bất kỳ trong [-10^5, 10^5]   : 6 cách x 300 chuỗi lệnh -> khớp oracle


nhiều số trùng trong [-5, 5]    : 6 cách x 300 chuỗi lệnh -> khớp oracle


mọi số trong [0, 100]           : 7 cách x 300 chuỗi lệnh -> khớp oracle


99% trong [0, 100]              : 6 cách x 300 chuỗi lệnh -> khớp oracle


một nửa ngoài [0, 100]          : 6 cách x 300 chuỗi lệnh -> khớp oracle


cỡ lớn, số bất kỳ               : 50000 lời gọi, 20084 lần findMedian -> khớp cách 2


cỡ lớn, mọi số trong [0, 100]   : 50000 lời gọi, 19979 lần findMedian -> khớp cách 2


## 13. Bảng so sánh các cách

`n` = số phần tử hiện có, `m` = số phần tử ngoài `[0, 100]` (cách 5).

| Cách | Ý tưởng | `addNum` | `findMedian` | Bộ nhớ | Nhận xét |
|---|---|---|---|---|---|
| 1. Sort mỗi lần hỏi | list thô + `sorted` khi hỏi | `O(1)` | `O(n log n)` | `O(n)` | dễ nghĩ nhất, dễ TLE |
| 2. List luôn sắp xếp | `bisect.insort` | `O(n)` (tìm `O(log n)` + dời `O(n)`) | `O(1)` | `O(n)` | thường vẫn chạy kịp nhờ dời bằng C, nhưng Big-O chưa tốt |
| 3. Hai heap | max-heap `low` + min-heap `high` | `O(log n)` | `O(1)` | `O(n)` | **cách chính**, câu trả lời phỏng vấn chuẩn |
| 3b. Hai heap bản gọn | `heappushpop` đi vòng qua `low` | `O(log n)` | `O(1)` | `O(n)` | ít nhánh `if` hơn, luôn tốn 2–3 thao tác heap |
| 4. Mảng đếm | `count[101]` + đếm tích lũy | `O(1)` | `O(101)` | `O(101)` | follow-up 1: chỉ khi **mọi** số trong `[0, 100]` |
| 5. Đếm + 2 vùng ngoài | `count[101]` + `below` / `above` | `O(1)` trong khoảng, `O(m)` ngoài khoảng | `O(101)` (thường gặp) | `O(101 + m)` | follow-up 2: 99% trong `[0, 100]`; vẫn đúng với mọi input |

Để cảm nhận sự khác biệt, cell dưới **ước lượng số bước cơ bản** theo đúng công thức của từng cách, cho kịch bản xấu: `addNum` và `findMedian` xen kẽ nhau (`q / 2` lần mỗi loại). Đây là đếm theo công thức, không đo thời gian thật, nên kết quả luôn cố định.

Hãy chú ý dòng `q = 50,000`: cách 1 cần cỡ **vài tỷ** bước, cách 2 cỡ **trăm triệu**, còn cách 3 chỉ khoảng **một triệu**. Cột mảng đếm không nhỏ hơn cột hai heap — vì duyệt 101 ô mỗi lần hỏi là hằng số nhưng không hề nhỏ. Ưu điểm thật của mảng đếm là `addNum` `O(1)`, bộ nhớ chỉ 101 ô và **không lớn lên** khi `n` tăng (stream có `10^9` số thì vẫn 101 ô).

In [25]:
import math


def estimate_steps(q):
    # Kịch bản xấu: q lời gọi, addNum và findMedian xen kẽ -> q // 2 lần mỗi loại
    adds = q // 2
    c1 = c2 = c3 = 0.0
    for n in range(1, adds + 1):           # lần hỏi thứ n diễn ra khi có n phần tử
        log_n = max(1.0, math.log2(n))
        c1 += 1 + n * log_n                # add O(1) + sort n số khi hỏi
        c2 += log_n + n / 2 + 1            # tìm vị trí + dời trung bình n/2 + hỏi O(1)
        c3 += 3 * log_n + 1                # tối đa 3 thao tác heap + hỏi O(1)
    c4 = adds * (1 + 101)                  # add O(1) + hỏi duyệt tối đa 101 ô
    return c1, c2, c3, c4


print(f"{'q':>7} | {'1.sort mỗi lần':>15} {'2.insort':>13} {'3.hai heap':>11} {'4.mảng đếm':>11}")
for q in [10**3, 10**4, 5 * 10**4]:
    c1, c2, c3, c4 = estimate_steps(q)
    print(f"{q:>7,} | {c1:>15,.0f} {c2:>13,.0f} {c3:>11,.0f} {c4:>11,.0f}")

      q |  1.sort mỗi lần      2.insort  3.hai heap  4.mảng đếm
  1,000 |       1,033,298        66,893      11,805      51,000
 10,000 |     144,615,282     6,310,484     167,701     510,000
 50,000 |   4,340,299,171   156,610,433   1,012,550   2,550,000


## 14. Các lỗi người mới hay gặp

Để thấy lỗi "tận mắt", ta viết các class lỗi bằng **kế thừa** (inheritance): `class BugX(MedianFinderTwoHeaps):` nghĩa là `BugX` dùng lại mọi method của cách 3, chỉ viết lại đúng method bị sai. Mỗi lỗi được chạy trên một chuỗi lệnh và so với oracle.

### Lỗi 1: Quên đổi dấu khi đọc max-heap

```python
return float(self.low[0])                    # SAI: low lưu số âm
return (self.low[0] + self.high[0]) / 2      # SAI
```

`low` lưu `-x`, nên đọc thẳng `self.low[0]` sẽ ra số âm (ví dụ `-2` thay vì `2`). Sửa: nhớ quy tắc **"vào âm, ra âm"** — push `-x`, xem `-low[0]`, lấy ra `-heappop(low)`.

### Lỗi 2: Chỉ lo kích thước, không so sánh với đỉnh `low`

```python
if len(self.low) == len(self.high):          # SAI: luân phiên đưa vào low / high
    heapq.heappush(self.low, -num)
else:
    heapq.heappush(self.high, num)
```

Kích thước luôn "đẹp" (I2 đúng) nhưng **I1 bị vỡ**: số lớn có thể nằm trong `low`. Với `5, 15, 1, 3, 8`: `low = {5, 1, 8}`, `high = {15, 3}` → median = `8` thay vì `5`. Sửa: bước 1 phải so `num <= -low[0]`.

### Lỗi 3: So sánh đúng nhưng quên bước cân bằng

Stream tăng dần `1, 2, 3, 4, 5`: mọi số sau số đầu đều lớn hơn đỉnh `low` → dồn hết vào `high` → `low = {1}`, `high = {2, 3, 4, 5}` → I2 vỡ, median sai. Sửa: luôn có bước 2 với hai nhánh `if / elif`.

In [26]:
def demo_bug(title, cls, tokens):
    ops, args = build_commands(tokens)
    got = [x for x in run_commands(cls, ops, args) if x is not None]
    right = [x for x in oracle_outputs(tokens) if x is not None]
    print(f"{title:<34} got = {got}  | đúng = {right}")


class BugForgetNegate(MedianFinderTwoHeaps):
    def findMedian(self):
        if len(self.low) > len(self.high):
            return float(self.low[0])                  # SAI: quên đổi dấu
        return (self.low[0] + self.high[0]) / 2        # SAI


class BugNoCompare(MedianFinderTwoHeaps):
    def addNum(self, num):
        if len(self.low) == len(self.high):            # SAI: không so với đỉnh low
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)


class BugNoRebalance(MedianFinderTwoHeaps):
    def addNum(self, num):
        if not self.low or num <= -self.low[0]:
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)
        # SAI: quên bước cân bằng kích thước


demo_bug("Lỗi 1 (quên đổi dấu), [1,2,3]", BugForgetNegate, [1, 2, "?", 3, "?"])
demo_bug("Lỗi 2 (không so sánh), 5,15,1,3,8", BugNoCompare, [5, 15, 1, 3, 8, "?"])
demo_bug("Lỗi 3 (quên cân bằng), 1..5", BugNoRebalance, [1, 2, 3, 4, 5, "?"])

Lỗi 1 (quên đổi dấu), [1,2,3]      got = [0.5, -2.0]  | đúng = [1.5, 2.0]
Lỗi 2 (không so sánh), 5,15,1,3,8  got = [8.0]  | đúng = [5.0]
Lỗi 3 (quên cân bằng), 1..5        got = [1.5]  | đúng = [3.0]


### Lỗi 4: Dùng `//` khi lấy trung bình

```python
return (-self.low[0] + self.high[0]) // 2    # SAI
```

`[1, 2]` → ra `1` thay vì `1.5`; `[-1, -2]` → ra `-2` thay vì `-1.5` (vì `//` làm tròn xuống). Sửa: dùng `/`.

### Lỗi 5: Đọc `self.low[0]` khi `low` còn rỗng

```python
if num <= -self.low[0]:                       # SAI ở lần addNum đầu tiên
```

Lần thêm đầu tiên `low = []` → `IndexError: list index out of range`. Sửa: `if not self.low or num <= -self.low[0]:` — và `not self.low` phải đứng **trước** `or`.

### Lỗi 6: Khai báo list ở mức class thay vì trong `__init__`

```python
class MedianFinder:
    low = []          # SAI: biến của CLASS, mọi object dùng chung một list
    high = []
```

Khi chạy riêng 1 test thì đúng, nhưng LeetCode chạy **nhiều test case liên tiếp**, mỗi test tạo object mới. Vì `low`, `high` thuộc về **class** nên object mới vẫn thấy dữ liệu cũ → test case sau bị "dính" dữ liệu của test case trước. Sửa: tạo `self.low = []`, `self.high = []` **bên trong** `__init__`.

### Lỗi 7 (follow-up): Lệch 1 khi đếm tích lũy

```python
if cum > k:           # SAI: phải là cum >= k
```

Với `[2, 2, 2, 5, 7, 9]` và `k = 3`: ở ô `2` thì `cum = 3`, điều kiện `3 > 3` sai nên bỏ qua → trả `5` thay vì `2`. Với `k = 6` (phần tử lớn nhất) còn tệ hơn: `cum` không bao giờ vượt quá 6 → hàm trả `None`. Sửa: `k` tính từ 1 và điều kiện là `cum >= k`.

In [27]:
class BugFloorDiv(MedianFinderTwoHeaps):
    def findMedian(self):
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) // 2      # SAI: //


class BugNoEmptyCheck(MedianFinderTwoHeaps):
    def addNum(self, num):
        if num <= -self.low[0]:                        # SAI: low có thể rỗng
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)


class BugSharedState(MedianFinderTwoHeaps):
    low = []        # SAI: biến của CLASS -> mọi object dùng CHUNG
    high = []

    def __init__(self):
        pass        # không tạo list riêng cho từng object


def kth_bug(count, k):
    cum = 0
    for value in range(len(count)):
        cum += count[value]
        if cum > k:                                    # SAI: phải là >=
            return value


demo_bug("Lỗi 4 (//), [1, 2]", BugFloorDiv, [1, 2, "?"])
demo_bug("Lỗi 4 (//), [-1, -2]", BugFloorDiv, [-1, -2, "?"])

try:
    BugNoEmptyCheck().addNum(1)
except IndexError as e:
    print("Lỗi 5 -> IndexError:", e)

print("Lỗi 6, test case 1:", show(run_commands(BugSharedState, ops1, args1)))
ops_t2, args_t2 = build_commands([10, "?"])
print("Lỗi 6, test case 2:", show(run_commands(BugSharedState, ops_t2, args_t2)), "| đúng: [null, null, 10.0]")

cnt = [0] * 101
for x in [2, 2, 2, 5, 7, 9]:
    cnt[x] += 1
print("Lỗi 7: kth_bug(k=3) =", kth_bug(cnt, 3), "| đúng:", kth_from_counts(cnt, 3))
print("Lỗi 7: kth_bug(k=6) =", kth_bug(cnt, 6), "| đúng:", kth_from_counts(cnt, 6))

Lỗi 4 (//), [1, 2]                 got = [1]  | đúng = [1.5]
Lỗi 4 (//), [-1, -2]               got = [-2]  | đúng = [-1.5]
Lỗi 5 -> IndexError: list index out of range
Lỗi 6, test case 1: [null, null, null, 1.5, null, 2.0]
Lỗi 6, test case 2: [null, null, 2.5] | đúng: [null, null, 10.0]
Lỗi 7: kth_bug(k=3) = 5 | đúng: 2
Lỗi 7: kth_bug(k=6) = None | đúng: 9


### Bảng tóm tắt lỗi

| Lỗi | Triệu chứng | Cách sửa |
|---|---|---|
| 1. Quên đổi dấu max-heap | median âm hoặc sai dấu | `-self.low[0]`, `-heapq.heappop(self.low)` |
| 2. Không so với đỉnh `low` | kích thước đúng mà median sai | `if not self.low or num <= -self.low[0]` |
| 3. Quên cân bằng | sai với stream tăng dần / giảm dần | hai nhánh `if / elif` sau khi đặt số |
| 4. `//` thay cho `/` | mất phần `.5`, sai với số âm | dùng `/` |
| 5. Không kiểm tra `low` rỗng | `IndexError` ở lần thêm đầu tiên | `not self.low or ...` (đặt trước `or`) |
| 6. Trạng thái ở mức class | test case sau "dính" dữ liệu test trước | khởi tạo `self.low = []` trong `__init__` |
| 7. Lệch 1 khi đếm tích lũy | median lệch sang giá trị kế tiếp, hoặc `None` | `k` tính từ 1, điều kiện `cum >= k` |

## 15. Mẹo nhận dạng pattern, biến thể và bài liên quan

### Dấu hiệu nhận dạng

- Đề nói **"data stream"**, "thêm từng phần tử", và hỏi **median / phần tử ở giữa / percentile** bất cứ lúc nào → nghĩ ngay tới **hai heap** (max-heap nửa dưới + min-heap nửa trên).
- Hỏi **phần tử lớn thứ `k`** của stream với `k` cố định → chỉ cần **một heap** kích thước `k` (bài 703, xem Bài tập 5).
- **Miền giá trị nhỏ** (ví dụ `[0, 100]`, `[0, 255]`) → **mảng đếm + đếm tích lũy**, bộ nhớ không phụ thuộc `n`.
- Cần cả **thêm và xóa** phần tử bất kỳ (cửa sổ trượt) → list sắp xếp + `bisect` (`O(k)` mỗi bước), hoặc hai heap + "xóa lười" (lazy deletion), hoặc cấu trúc cây cân bằng / `SortedList` (thư viện `sortedcontainers` có sẵn trên LeetCode nhưng không thuộc thư viện chuẩn).

### Template trong đầu

```text
low  = max-heap (lưu số âm)  -> nửa nhỏ, đỉnh = số lớn nhất nửa nhỏ
high = min-heap              -> nửa lớn, đỉnh = số nhỏ nhất nửa lớn
add(x):  đặt x vào đúng nửa (so với đỉnh low)  ->  cân bằng (chênh tối đa 1)
median:  lẻ -> đỉnh low ; chẵn -> trung bình hai đỉnh
```

### Biến thể: percentile bất kỳ

Median là **percentile 50%**. Muốn percentile `p` (ví dụ `p = 0.9`, giá trị mà 90% dữ liệu nhỏ hơn hoặc bằng), vẫn dùng hai heap nhưng đổi luật cân bằng: giữ `len(low) = ceil(p · n)` thay vì `ceil(n / 2)`. Đỉnh của `low` khi đó là percentile cần tìm. Đây chính là cách nhiều hệ thống giám sát tính "độ trễ p90 / p99" theo thời gian thực.

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 480. Sliding Window Median (Hard) | median của cửa sổ trượt: cần thêm **và xóa** → hai heap + xóa lười, hoặc list sắp xếp + `bisect` (Bài tập 6) |
| 4. Median of Two Sorted Arrays (Hard) | median của dữ liệu **tĩnh** đã sắp xếp: binary search trên vết cắt chia đôi — cùng ý "nửa dưới / nửa trên" |
| 703. Kth Largest Element in a Stream (Easy) | một min-heap kích thước `k` trên stream (Bài tập 5) |
| 502. IPO (Hard) | hai heap phối hợp: một heap theo vốn yêu cầu, một heap theo lợi nhuận |
| 1093. Statistics from a Large Sample (Medium) | dữ liệu cho sẵn dạng **mảng đếm** `count[0..255]`, tìm median bằng đếm tích lũy — đúng follow-up 1 |
| 462. Minimum Moves to Equal Array Elements II (Medium) | median là điểm làm tổng khoảng cách nhỏ nhất |
| 1825. Finding MK Average (Hard) | stream + cửa sổ + cần các phần tử theo thứ tự → nhiều cấu trúc sắp xếp phối hợp |

## 16. Bài tập vận dụng

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý ở mục 17, đáp án ở mục 18 (cuối notebook) — hãy tự làm trước!

### Bài 1 — Đoán output (không chạy code)

```python
import heapq
low = []
for x in [3, 9, 1]:
    heapq.heappush(low, -x)
print(low[0], -low[0])
print(-heapq.heappop(low))
print(len(low), -low[0])
print((3 + 4) / 2, (3 + 4) // 2)
print(sorted([8, 2, 6, 4])[4 // 2])
```

Viết ra 5 dòng output và giải thích từng dòng. Câu hỏi thêm: dòng cuối có phải median của `[8, 2, 6, 4]` không?

### Bài 2 — Tính tay bảng trạng thái hai heap

Cho stream `4, 10, 2, 7, 6, 1` và `findMedian` được gọi sau **mỗi** lần thêm. Lập bảng giống mục 8.1 với các cột: `num`, bước 1 (so với đỉnh `low`, vào heap nào), kích thước sau bước 1, bước 2 (cân bằng gì), `low`, `high`, median.

Ở những bước nào có cân bằng? Theo chiều nào? Median cuối cùng là bao nhiêu?

### Bài 3 — Điền TODO: hai heap

Hoàn thiện class dưới đây theo đúng cách 3. Đừng quên quy tắc "vào âm, ra âm" và trường hợp `low` rỗng.

In [28]:
class MedianFinderExercise:
    def __init__(self):
        # TODO 1: tạo self.low (max-heap, lưu số âm) và self.high (min-heap)
        pass

    def addNum(self, num: int) -> None:
        # TODO 2: nếu low rỗng hoặc num <= max của low -> push -num vào low; ngược lại push num vào high
        # TODO 3: nếu len(low) > len(high) + 1 -> chuyển đỉnh low sang high
        # TODO 4: nếu len(high) > len(low)     -> chuyển đỉnh high về low
        pass

    def findMedian(self) -> float:
        # TODO 5: lẻ -> đỉnh low (đổi dấu, float); chẵn -> trung bình hai đỉnh (dùng /)
        pass


# Tự làm xong rồi mới mở test:
# ops, args = build_commands([1, 2, "?", 3, "?"])
# assert run_commands(MedianFinderExercise, ops, args) == [None, None, None, 1.5, None, 2.0]
# check_cases(MedianFinderExercise, basic_cases)

### Bài 4 — Tự viết lại: follow-up 1 bằng mảng đếm

Không nhìn lại mục 10, tự viết class cho trường hợp **mọi** số nằm trong `[0, 100]`:

- `addNum` phải là `O(1)`;
- bộ nhớ chỉ được dùng một mảng 101 ô và vài biến đếm (không lưu từng số);
- `findMedian` dùng đếm tích lũy, xử lý đúng cả `n` lẻ và `n` chẵn.

In [29]:
class MedianFinder0To100Exercise:
    def __init__(self):
        # TODO: mảng đếm 101 ô + tổng số phần tử
        raise NotImplementedError

    def addNum(self, num: int) -> None:
        # TODO
        raise NotImplementedError

    def findMedian(self) -> float:
        # TODO: tìm phần tử thứ k (k tính từ 1) bằng đếm tích lũy
        raise NotImplementedError


# Gợi ý test:
# ops, args = build_commands([2, 7, 2, 5, 9, 2, "?"])
# assert run_commands(MedianFinder0To100Exercise, ops, args)[-1] == 3.5
# check_cases(MedianFinder0To100Exercise, counting_cases)

### Bài 5 — Biến thể: LeetCode 703. Kth Largest Element in a Stream

Thiết kế class `KthLargest`:

- `KthLargest(int k, int[] nums)` khởi tạo với số nguyên `k` và các số ban đầu `nums`.
- `int add(int val)` thêm `val` rồi trả về **số lớn thứ `k`** trong tất cả các số hiện có (lớn thứ `k` theo thứ tự đã sắp xếp, tính cả số trùng — không phải "giá trị khác nhau thứ `k`").

```text
Input
["KthLargest", "add", "add", "add", "add", "add"]
[[3, [4, 5, 8, 2]], [3], [5], [10], [9], [4]]
Output
[null, 4, 5, 5, 8, 8]
```

Đề đảm bảo khi cần trả lời thì luôn có ít nhất `k` số. Yêu cầu: `add` tốn `O(log k)`, bộ nhớ `O(k)` (không giữ cả stream). Hàm `run_commands` dùng được luôn: `args[0] = [3, [4, 5, 8, 2]]` nên `cls(*args[0])` chính là `KthLargest(3, [4, 5, 8, 2])`.

In [30]:
from typing import List


class KthLargestExercise:
    def __init__(self, k: int, nums: List[int]):
        # TODO: lưu k và một min-heap chỉ chứa k số lớn nhất
        pass

    def add(self, val: int) -> int:
        # TODO: push val; nếu heap dài hơn k thì pop; trả về đỉnh heap
        pass


# Gợi ý test:
# ops = ["KthLargest", "add", "add", "add", "add", "add"]
# args = [[3, [4, 5, 8, 2]], [3], [5], [10], [9], [4]]
# assert run_commands(KthLargestExercise, ops, args) == [None, 4, 5, 5, 8, 8]

### Bài 6 — Mini challenge: LeetCode 480. Sliding Window Median (bản `O(n·k)`)

Cho mảng `nums` và số `k`. Một cửa sổ dài `k` trượt từ trái sang phải, mỗi lần 1 bước. Trả về list median của từng cửa sổ (dạng `float`).

```text
nums = [1,3,-1,-3,5,3,6,7],  k = 3  ->  [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
nums = [1,2,3,4,2,3,1,4,2],  k = 3  ->  [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
nums = [1,4,2,3],            k = 2  ->  [2.5, 3.0, 2.5]
```

Yêu cầu: **không** sắp xếp lại từng cửa sổ (`O(n·k log k)`). Hãy giữ cửa sổ là một list đã sắp xếp; mỗi bước trượt **thêm** 1 số (`insort`) và **xóa** 1 số (tìm vị trí bằng `bisect_left` rồi `pop`) → `O(n·k)`. (Bản `O(n log k)` dùng hai heap + xóa lười — tìm hiểu thêm sau khi làm xong.)

In [31]:
def median_sliding_window_exercise(nums: List[int], k: int) -> List[float]:
    # TODO 1: window = sorted(k số đầu), kết quả bắt đầu bằng median của window
    # TODO 2: với mỗi i từ k tới cuối: xóa nums[i - k] khỏi window, insort nums[i]
    # TODO 3: thêm median của window vào kết quả
    pass


# Gợi ý test:
# assert median_sliding_window_exercise([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
# assert median_sliding_window_exercise([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
# assert median_sliding_window_exercise([1, 4, 2, 3], 2) == [2.5, 3.0, 2.5]

## 17. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Heap số âm: đỉnh là số âm nhỏ nhất = số thật lớn nhất. | Sau 3 lần push, `low` chứa `-3, -9, -1` và `low[0] = -9`. | Pop lấy `-9` ra → in `9`; còn lại `-3, -1`. `sorted` ra `[2, 4, 6, 8]`. |
| 2 | Mỗi bước: so với đỉnh `low` → đặt → cân bằng. | Sau bước 1, hỏi: `len(low) > len(high) + 1`? hay `len(high) > len(low)`? | Bước 5 (thêm `6`): `6 > 4` → `high = {6, 7, 10}` → `high` nhiều hơn → chuyển `6` về `low`. |
| 3 | 2 bước: đặt đúng nửa, rồi cân bằng. | `if not self.low or num <= -self.low[0]:` | `heapq.heappush(self.high, -heapq.heappop(self.low))` và chiều ngược lại. |
| 4 | Lưu **số lần xuất hiện**, không lưu từng số. | `self.count = [0] * 101`, `self.n = 0`. | Hàm `_kth(k)`: cộng dồn `cum`, `if cum >= k: return v`; lẻ thì `k = n // 2 + 1`. |
| 5 | Chỉ cần giữ `k` số lớn nhất; số **nhỏ nhất** trong `k` số đó là đáp án. | Min-heap kích thước `k`; dư thì pop số nhỏ nhất. | `heappush(h, val)`; `if len(h) > k: heappop(h)`; `return h[0]`. Trong `__init__` gọi `add` cho từng số của `nums`. |
| 6 | Cửa sổ là list sắp xếp dài `k`; trượt = bỏ 1 số cũ, thêm 1 số mới. | Xóa: `window.pop(bisect.bisect_left(window, old))`; thêm: `bisect.insort(window, new)`. | Mỗi bước lấy `median_of_sorted(window)`. |

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
-9 9
9
2 3
3.5 3
6
```

- Push `-3, -9, -1` → số âm nhỏ nhất là `-9` nằm ở gốc: `low[0] = -9`, đổi dấu `-low[0] = 9` = số lớn nhất của `{3, 9, 1}`.
- `-heappop(low)`: lấy `-9` ra rồi đổi dấu → `9`.
- Còn lại 2 phần tử (`-3`, `-1`), đỉnh mới là `-3` → số lớn nhất còn lại là `3`.
- `/` cho `3.5`, còn `//` cho `3`.
- `sorted([8, 2, 6, 4]) = [2, 4, 6, 8]`, chỉ số `4 // 2 = 2` → `6`. Nhưng đây **không** phải median: `n = 4` chẵn nên median = `(4 + 6) / 2 = 5.0`. Lấy `a[n // 2]` chỉ đúng khi `n` lẻ.

### Đáp án Bài 2

| Bước | `num` | Bước 1: so với đỉnh `low` | Kích thước sau bước 1 | Bước 2: cân bằng | `low` | `high` | median |
|---:|---:|---|---|---|---|---|---:|
| 1 | 4 | `low` rỗng → vào `low` | 1 / 0 | không cần | `[4]` | `[]` | `4.0` |
| 2 | 10 | `10 > 4` → vào `high` | 1 / 1 | không cần | `[4]` | `[10]` | `7.0` |
| 3 | 2 | `2 <= 4` → vào `low` | 2 / 1 | không cần | `[2, 4]` | `[10]` | `4.0` |
| 4 | 7 | `7 > 4` → vào `high` | 2 / 2 | không cần | `[2, 4]` | `[7, 10]` | `5.5` |
| 5 | 6 | `6 > 4` → vào `high` | 2 / 3 | `high` nhiều hơn → chuyển `6` về `low` | `[2, 4, 6]` | `[7, 10]` | `6.0` |
| 6 | 1 | `1 <= 6` → vào `low` | 4 / 2 | `low` dư → chuyển `6` sang `high` | `[1, 2, 4]` | `[6, 7, 10]` | `5.0` |

Có cân bằng ở bước 5 (`high` → `low`) và bước 6 (`low` → `high`); số `6` lại "đi qua đi lại" giống số `5` ở mục 8.1. Median cuối = `(4 + 6) / 2 = 5.0`, khớp với `[1, 2, 4, 6, 7, 10]`.

Cell kế tiếp chạy lại đoạn code Bài 1 và `trace_two_heaps` cho Bài 2 để kiểm chứng; cell sau nữa là **đáp án Bài 3–6** kèm assert.

In [32]:
# Kiểm chứng Bài 1
low = []
for x in [3, 9, 1]:
    heapq.heappush(low, -x)
print(low[0], -low[0])
print(-heapq.heappop(low))
print(len(low), -low[0])
print((3 + 4) / 2, (3 + 4) // 2)
print(sorted([8, 2, 6, 4])[4 // 2], "| median thật:", median_of_sorted(sorted([8, 2, 6, 4])))
print()

# Kiểm chứng Bài 2
assert trace_two_heaps([4, 10, 2, 7, 6, 1]) == 5.0

-9 9
9
2 3
3.5 3
6 | median thật: 5.0

Stream: [4, 10, 2, 7, 6, 1]
Bước 1: thêm 4  | low rỗng -> vào low | kích thước 1/0 | không cần cân bằng
   low = [4] (thô [-4]) | high = [] | median = 4.0
Bước 2: thêm 10 | 10 > đỉnh low 4 -> vào high | kích thước 1/1 | không cần cân bằng
   low = [4] (thô [-4]) | high = [10] | median = 7.0
Bước 3: thêm 2  | 2 <= đỉnh low 4 -> vào low | kích thước 2/1 | không cần cân bằng
   low = [2, 4] (thô [-4, -2]) | high = [10] | median = 4.0
Bước 4: thêm 7  | 7 > đỉnh low 4 -> vào high | kích thước 2/2 | không cần cân bằng
   low = [2, 4] (thô [-4, -2]) | high = [7, 10] | median = 5.5
Bước 5: thêm 6  | 6 > đỉnh low 4 -> vào high | kích thước 2/3 | high nhiều hơn -> chuyển 6 về low
   low = [2, 4, 6] (thô [-6, -2, -4]) | high = [7, 10] | median = 6.0
Bước 6: thêm 1  | 1 <= đỉnh low 6 -> vào low | kích thước 4/2 | low dư -> chuyển 6 sang high
   low = [1, 2, 4] (thô [-4, -2, -1]) | high = [6, 7, 10] | median = 5.0


In [33]:
from typing import List


class AnswerMedianFinderExercise:
    def __init__(self):
        self.low = []     # max-heap (số âm)
        self.high = []    # min-heap

    def addNum(self, num: int) -> None:
        if not self.low or num <= -self.low[0]:
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)
        if len(self.low) > len(self.high) + 1:
            heapq.heappush(self.high, -heapq.heappop(self.low))
        elif len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


class AnswerMedianFinder0To100:
    def __init__(self):
        self.count = [0] * 101
        self.n = 0

    def addNum(self, num: int) -> None:
        self.count[num] += 1
        self.n += 1

    def _kth(self, k):
        cum = 0
        for value in range(101):
            cum += self.count[value]
            if cum >= k:
                return value

    def findMedian(self) -> float:
        if self.n % 2 == 1:
            return float(self._kth(self.n // 2 + 1))
        return (self._kth(self.n // 2) + self._kth(self.n // 2 + 1)) / 2


class AnswerKthLargest:
    def __init__(self, k: int, nums: List[int]):
        self.k = k
        self.heap = []                    # min-heap chứa tối đa k số lớn nhất
        for x in nums:
            self.add(x)

    def add(self, val: int) -> int:
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)      # bỏ số nhỏ nhất, chỉ giữ k số lớn nhất
        return self.heap[0]               # nhỏ nhất trong k số lớn nhất = lớn thứ k


def answer_median_sliding_window(nums: List[int], k: int) -> List[float]:
    window = sorted(nums[:k])
    result = [median_of_sorted(window)]
    for i in range(k, len(nums)):
        old = nums[i - k]
        window.pop(bisect.bisect_left(window, old))     # bỏ số rời cửa sổ
        bisect.insort(window, nums[i])                  # thêm số mới vào cửa sổ
        result.append(median_of_sorted(window))
    return result


# Bài 3
assert run_commands(AnswerMedianFinderExercise, ops1, args1) == [None, None, None, 1.5, None, 2.0]
check_cases(AnswerMedianFinderExercise, basic_cases)

# Bài 4
check_cases(AnswerMedianFinder0To100, counting_cases)

# Bài 5
ops_k = ["KthLargest", "add", "add", "add", "add", "add"]
args_k = [[3, [4, 5, 8, 2]], [3], [5], [10], [9], [4]]
assert run_commands(AnswerKthLargest, ops_k, args_k) == [None, 4, 5, 5, 8, 8]
ops_k2 = ["KthLargest", "add", "add", "add", "add"]
args_k2 = [[4, [7, 7, 7, 7, 8, 3]], [2], [10], [9], [9]]
assert run_commands(AnswerKthLargest, ops_k2, args_k2) == [None, 7, 7, 7, 8]

# Bài 6
assert answer_median_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1.0, -1.0, -1.0, 3.0, 5.0, 6.0]
assert answer_median_sliding_window([1, 2, 3, 4, 2, 3, 1, 4, 2], 3) == [2.0, 3.0, 3.0, 3.0, 2.0, 3.0, 2.0]
assert answer_median_sliding_window([1, 4, 2, 3], 2) == [2.5, 3.0, 2.5]

# Đối chiếu thêm với input ngẫu nhiên (seed cố định)
random.seed(2026)
for _ in range(200):
    tokens = random_tokens(random.randint(1, 30), lambda: random.randint(0, 100))
    ops, args = build_commands(tokens)
    expected = oracle_outputs(tokens)
    assert same_outputs(run_commands(AnswerMedianFinderExercise, ops, args), expected)
    assert same_outputs(run_commands(AnswerMedianFinder0To100, ops, args), expected)

    k = random.randint(1, 5)
    nums = [random.randint(-9, 9) for _ in range(random.randint(k, 10))]
    kl = AnswerKthLargest(k, nums[:k])
    seen = nums[:k]
    for v in nums[k:]:
        seen.append(v)
        assert kl.add(v) == sorted(seen, reverse=True)[k - 1]

    window_k = random.randint(1, len(nums))
    brute = [median_of_sorted(sorted(nums[i:i + window_k])) for i in range(len(nums) - window_k + 1)]
    assert answer_median_sliding_window(nums, window_k) == brute

print("Đáp án tham khảo chạy PASS toàn bộ test!")

AnswerMedianFinderExercise: 9 test PASS
AnswerMedianFinder0To100: 10 test PASS
Đáp án tham khảo chạy PASS toàn bộ test!


## 19. Template cuối cùng nên nhớ

**Hai heap (cách chính)** — `addNum` `O(log n)`, `findMedian` `O(1)`, bộ nhớ `O(n)`:

```python
import heapq

class MedianFinder:
    def __init__(self):
        self.low = []    # max-heap (lưu số âm): nửa nhỏ
        self.high = []   # min-heap: nửa lớn

    def addNum(self, num: int) -> None:
        if not self.low or num <= -self.low[0]:
            heapq.heappush(self.low, -num)
        else:
            heapq.heappush(self.high, num)
        if len(self.low) > len(self.high) + 1:
            heapq.heappush(self.high, -heapq.heappop(self.low))
        elif len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2
```

**Câu thần chú:** *"Nửa dưới max-heap (số âm), nửa trên min-heap; đặt đúng nửa → cân bằng → median nằm ở đỉnh."*

**Hai invariant:** max của `low` ≤ min của `high`; `len(low) == len(high)` hoặc `len(low) == len(high) + 1`.

**Follow-up `[0, 100]`:** *"Đếm 101 ô, cộng dồn tới khi `cum >= k`; lẻ lấy `k = n // 2 + 1`, chẵn lấy `k = n // 2` và `n // 2 + 1`."*

**Follow-up 99%:** *"Thêm hai vùng ngoài khoảng (`below`, `above`, giữ sắp xếp); tìm phần tử thứ `k` lần lượt qua `below` → mảng đếm → `above`."*

## 20. Checklist tự kiểm tra trước khi submit

- Tên class và method đúng y như đề: `MedianFinder`, `addNum`, `findMedian`?
- Mọi trạng thái (`self.low`, `self.high`) được tạo **trong `__init__`**, không phải ở mức class?
- Push vào `low` bằng `-num`, đọc bằng `-self.low[0]`, lấy ra bằng `-heapq.heappop(self.low)` ("vào âm, ra âm")?
- Điều kiện bước 1 có `not self.low or ...` (đặt `not self.low` **trước** `or`) để không lỗi ở lần thêm đầu tiên?
- Bước 2 có **đủ cả hai** nhánh cân bằng (`low` dư 2, `high` nhiều hơn `low`)?
- `findMedian` khớp **đúng quy ước** của `addNum` (`low` giữ phần dư khi `n` lẻ)?
- Trường hợp chẵn dùng `/` chứ không phải `//`, và kết quả là `float`?
- Đã test: Example 1, một phần tử, hỏi liên tiếp, số âm, số trùng, stream tăng dần / giảm dần, biên `±10^5`?
- Nếu làm follow-up: `k` tính từ 1, điều kiện `cum >= k`, xử lý cả hai phần tử giữa nằm ở hai ô khác nhau, và (bản 99%) median rơi ra ngoài `[0, 100]`?